# AInstein component: LLM QA

In [3]:
import sys
from pathlib import Path

path_project = Path.cwd().parent
sys.path.append(str(path_project))

In [4]:
from AInstein import (
    get_llm_azure_openai, # conexión al modelo GPT
    get_settings, # conexión a las configuraciones de los modelos
    BigQueryManager
)

In [5]:
import re
import json
from collections import defaultdict

import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML
from openpyxl import load_workbook
from io import BytesIO
import copy

# Settings

In [7]:
# Settings
ENVIRONMENT: str = 'bdb-gcp-sbx-ia'
WORKPLACE_PROJECT_ID: str = 'geo-cargas_laborales'

# Obtener las configuraciones del proyecto
settings = get_settings(WORKPLACE_PROJECT_ID, environment=ENVIRONMENT)

# Models

In [8]:
# Crear instancias de los modelos
llm = get_llm_azure_openai(settings)  # Modelo de Azure OpenAI

# Responses

## General

In [9]:
import re
import json
import copy
import os
from io import BytesIO
from openpyxl import load_workbook

# ===========================================================================
#  ProcesadorTranscripcionTeams
#  v7 — Búsqueda de colaborador en Excel + recomendaciones IA automáticas
# ===========================================================================

class ProcesadorTranscripcionTeams:
    """
    CAMBIOS v7
    ----------
    1. capturar_contexto_conversacional ahora busca al colaborador en un
       archivo Excel usando su número de documento. Extrae automáticamente
       cargo, vicepresidencia y gerencia. Ya no se ingresan manualmente.
       Fallback manual disponible si el archivo no existe o no se encuentra
       al colaborador.

    2. Las partes donde el Agente da recomendaciones ahora las aplican
       automáticamente sin dar opción de corrección al usuario:
         - _mostrar_incoherencia_y_corregir: aplica los cambios sugeridos
           de forma silenciosa e informa el resultado.
         - _manejar_alerta_relevancia: si es irrelevante → descarta
           automáticamente; si el nombre es poco representativo → aplica
           el primer nombre sugerido automáticamente.
       Las partes donde el usuario ingresa información que solo él conoce
       (frecuencia, volumen, autonomía, proceso del área, nombre) NO cambian.

    CAMBIOS ADICIONALES
    -------------------
    - capturar_contexto_conversacional: si el número de documento NO se
      encuentra en la planta, ya NO ofrece ingreso manual. Solo permite
      reintentar con otro número de documento.
    - _editar_actividad_individual: el usuario escoge el campo a editar
      mediante número, no escribiendo el nombre de la columna.
    """

    # -----------------------------------------------------------------------
    # CONSTANTES COMPARTIDAS
    # -----------------------------------------------------------------------
    FRECUENCIAS_VALIDAS = [
        "Diario", "Semanal", "Quincenal",
        "Mensual", "Bimensual", "Trimestral", "Semestral", "Anual"
    ]

    # -----------------------------------------------------------------------
    # 1. LECTURA Y LIMPIEZA DEL VTT
    # -----------------------------------------------------------------------
    def leer_archivo_vtt(self, contenido):
        conversaciones = []
        lineas = contenido.splitlines()

        timestamp_actual = None
        buffer_texto = []

        for linea in lineas:
            linea = linea.strip()

            if (
                not linea
                or linea == "WEBVTT"
                or re.match(r"^[a-f0-9\-]+\/\d+\-\d+$", linea)
            ):
                continue

            if "-->" in linea:
                if timestamp_actual and buffer_texto:
                    conversaciones.append({
                        "timestamp": timestamp_actual,
                        "texto": " ".join(buffer_texto).strip()
                    })
                    buffer_texto = []

                inicio = linea.split("-->")[0].strip()
                timestamp_actual = inicio.split(".")[0]
                continue

            linea = re.sub(r"<v[^>]*>", "", linea)
            linea = re.sub(r"</v>", "", linea)
            linea = re.sub(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ\s,]+:\s*", "", linea)

            if linea:
                buffer_texto.append(linea)

        if timestamp_actual and buffer_texto:
            conversaciones.append({
                "timestamp": timestamp_actual,
                "texto": " ".join(buffer_texto).strip()
            })

        return conversaciones

    # -----------------------------------------------------------------------
    # 2. EXTRACCIÓN DE ACTIVIDADES (SIN IA)
    # -----------------------------------------------------------------------
    def extraer_actividades(self, conversaciones):
        actividades = []
        actividad_actual = None

        patrones_inicio = [
            r"\binicio\b", r"\biniciar\b", r"\bcomienzo\b",
            r"\bempiezo\b", r"\bvoy a iniciar\b"
        ]
        patrones_fin = [
            r"\bfinalizo\b", r"\btermino\b",
            r"\bterminar\b", r"\bfinalizar\b"
        ]

        for conv in conversaciones:
            texto = conv["texto"].lower()
            timestamp = conv["timestamp"]

            if any(re.search(p, texto) for p in patrones_inicio):
                actividad_actual = {
                    "descripcion": conv["texto"],
                    "inicio": timestamp,
                    "fin": None
                }
            elif actividad_actual and any(re.search(p, texto) for p in patrones_fin):
                actividad_actual["fin"] = timestamp
                actividades.append(actividad_actual)
                actividad_actual = None

        return actividades

    # -----------------------------------------------------------------------
    # 3. CÁLCULO DE DURACIÓN
    # -----------------------------------------------------------------------
    def calcular_duracion_minutos(self, inicio, fin):
        h1, m1, s1 = map(int, inicio.split(":"))
        h2, m2, s2 = map(int, fin.split(":"))
        t1 = h1 * 3600 + m1 * 60 + s1
        t2 = h2 * 3600 + m2 * 60 + s2
        return max((t2 - t1) // 60, 0)

    # -----------------------------------------------------------------------
    # 4. PIPELINE DE PROCESAMIENTO
    # -----------------------------------------------------------------------
    def procesar_archivo(self, contenido):
        conversaciones = self.leer_archivo_vtt(contenido)
        actividades = self.extraer_actividades(conversaciones)

        resultado = []
        for a in actividades:
            if not a["fin"]:
                continue
            duracion = self.calcular_duracion_minutos(a["inicio"], a["fin"])
            if duracion <= 0:
                continue
            resultado.append({
                "actividad": a["descripcion"],
                "inicio": a["inicio"],
                "fin": a["fin"],
                "duracion_min": duracion
            })
        return resultado

    # -----------------------------------------------------------------------
    # 5. ENRIQUECIMIENTO CON IA (UNIDAD + PHVA)
    # -----------------------------------------------------------------------
    def _parse_json_seguro(self, texto):
        texto = texto.strip()
        if not texto:
            raise ValueError("❌ El LLM devolvió una respuesta vacía")
        if texto.startswith("```"):
            texto = re.sub(r"```json|```", "", texto).strip()
        inicio = min(
            [i for i in [texto.find("["), texto.find("{")] if i != -1],
            default=-1
        )
        if inicio > 0:
            texto = texto[inicio:]
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            print("❌ JSON inválido devuelto por el LLM")
            print("Respuesta cruda:")
            print(texto)
            raise

    def enriquecer_actividades(self, actividades, cargo, vicepresidencia):
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {cargo}
- Vicepresidencia: {vicepresidencia}

Para cada actividad, debes:
- unidad_medida (ej: solicitudes, informes, reuniones, casos, desarrollos)
- phva (Planear, Hacer, Verificar, Actuar)

⚠️ REGLAS ESTRICTAS:
- Devuelve EXCLUSIVAMENTE un JSON válido
- NO incluyas texto antes o después
- NO expliques nada
- NO uses markdown
- Devuelve una LISTA del mismo tamaño que la entrada

Formato exacto de salida:
[
  {{
    "nombre": "texto corto",
    "unidad_medida": "texto",
    "phva": "Planear | Hacer | Verificar | Actuar"
  }}
]

Actividades de entrada:
{json.dumps(actividades, indent=2, ensure_ascii=False)}
"""
        reply = llm.invoke(prompt)
        return self._parse_json_seguro(reply.content)

    # -----------------------------------------------------------------------
    # 5b. DETECCIÓN DE ACTIVIDADES IRRELEVANTES O QUE NECESITEN MÁS DETALLE
    # -----------------------------------------------------------------------
    def _evaluar_relevancia_y_detalle(self, actividad, cargo, vicepresidencia):
        nombre = actividad.get("nombre", "").strip()

        vars_operativas = {
            k: v for k, v in actividad.items()
            if k not in ("nombre", "descripcion", "_origen",
                         "_consolidada", "_ocurrencias",
                         "_duraciones_originales", "_analisis_interno")
            and v not in (None, "", [])
        }

        prompt = f"""
Eres un experto en Levantamiento de Cargas Laborales.

Analiza la siguiente actividad de un colaborador con:
- Cargo: {cargo}
- Vicepresidencia: {vicepresidencia}

Nombre de la actividad: "{nombre}"

Variables operativas registradas:
{json.dumps(vars_operativas, indent=2, ensure_ascii=False)}

Tu tarea es evaluar DOS cosas:

1. ¿Es esta actividad IRRELEVANTE para un levantamiento de cargas?
   Considera irrelevantes: actividades personales (ir al baño, tomar agua,
   almorzar), actividades fuera del ámbito laboral, actividades demasiado
   triviales que no representan carga laboral real.

2. ¿El NOMBRE es suficientemente representativo considerando TODO el
   contexto (frecuencia, volumen, duración, proceso del área, autonomía,
   observaciones y el cargo del colaborador)?
   - Si el nombre es genérico pero las variables operativas aclaran bien
     de qué se trata, NO marques como que necesita detalle.
   - Solo marca "necesita_detalle: true" si el nombre sigue siendo
     ambiguo o podría confundirse con otra actividad, incluso teniendo
     en cuenta el resto de variables.
   - Cuando necesita_detalle sea true, propón 2 o 3 nombres alternativos
     concretos y descriptivos que reflejen mejor lo que las variables
     sugieren. Los nombres deben ser cortos (máx. 8 palabras), claros y
     en el idioma español.

Responde EXCLUSIVAMENTE con un JSON válido:

{{
  "es_irrelevante": true | false,
  "necesita_detalle": true | false,
  "razon_irrelevante": "explicación breve o cadena vacía",
  "razon_detalle": "explicación breve de por qué el nombre no es representativo, o cadena vacía",
  "nombres_sugeridos": ["Nombre sugerido 1", "Nombre sugerido 2", "Nombre sugerido 3"]
}}

Si necesita_detalle es false, "nombres_sugeridos" debe ser [].
"""
        reply = llm.invoke(prompt)
        try:
            resultado = self._parse_json_seguro(reply.content)
            if "nombres_sugeridos" not in resultado:
                resultado["nombres_sugeridos"] = []
            return resultado
        except Exception:
            return {
                "es_irrelevante": False,
                "necesita_detalle": False,
                "razon_irrelevante": "",
                "razon_detalle": "",
                "nombres_sugeridos": [],
            }

    def _manejar_alerta_relevancia(self, actividad, evaluacion):
        hay_alerta = evaluacion["es_irrelevante"] or evaluacion["necesita_detalle"]
        if not hay_alerta:
            return "continuar", actividad

        print("\n" + "─" * 60)

        if evaluacion["es_irrelevante"]:
            print("⚠️  ACTIVIDAD NO RELEVANTE — descartada automáticamente")
            print("─" * 60)
            print(f"   {evaluacion['razon_irrelevante']}")
            print(
                "\n   ℹ️  El agente ha descartado esta actividad porque no\n"
                "   representa carga laboral real en el levantamiento."
            )
            print("─" * 60)
            return "descartar", None

        if evaluacion["necesita_detalle"]:
            sugeridos = [s for s in evaluacion.get("nombres_sugeridos", []) if s]
            print("📝  NOMBRE POCO REPRESENTATIVO — ajustado automáticamente")
            print("─" * 60)
            print(f"   {evaluacion['razon_detalle']}")

            if sugeridos:
                nombre_anterior = actividad.get("nombre", "")
                actividad["nombre"] = sugeridos[0]
                print(f"\n   Nombre anterior : \"{nombre_anterior}\"")
                print(f"   Nombre aplicado : \"{sugeridos[0]}\"")
                if len(sugeridos) > 1:
                    print(
                        f"\n   (Otras opciones descartadas: "
                        + ", ".join(f'"{s}"' for s in sugeridos[1:]) + ")"
                    )
                print(
                    "\n   ℹ️  El agente ha actualizado el nombre para que sea\n"
                    "   más representativo en el levantamiento."
                )
            else:
                print(
                    "\n   ℹ️  El agente no pudo generar un nombre alternativo.\n"
                    "   Se conserva el nombre original."
                )
            print("─" * 60)

        return "continuar", actividad

    # -----------------------------------------------------------------------
    # 6. CONSTRUCCIÓN DE RESUMEN ESTRUCTURADO
    # -----------------------------------------------------------------------
    def construir_resumen_actividades(self, actividades_enriquecidas):
        resumen = ""
        for i, act in enumerate(actividades_enriquecidas, start=1):
            resumen += f"""
Actividad {i}:
- Descripción: {act['nombre']}
- Frecuencia: {act.get('frecuencia', 'No especificada')}
- Duración (minutos): {act.get('duracion_min')}
- Proceso del área: {act.get('proceso_area')}
- Volumen: {act.get('volumen', 'N/A')}
- Unidad de Medida: {act.get('unidad_medida')}
- Tipo de actividad (PHVA): {act.get('phva')}
- Autonomía: {act.get('autonomia', 'N/A')}%
"""
            if act.get("observaciones"):
                resumen += f"- Observaciones: {act['observaciones']}\n"
        return resumen

    # -----------------------------------------------------------------------
    # 7. RESUMEN PARA CONFIRMACIÓN DEL USUARIO
    # -----------------------------------------------------------------------
    def resumen_para_confirmacion(self, actividades_enriquecidas, contexto):
        resumen_actividades = self.construir_resumen_actividades(
            actividades_enriquecidas
        )
        prompt = f"""
Eres un asistente de Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {contexto['cargo']}
- Vicepresidencia: {contexto['vicepresidencia']}

A continuación se presenta un resumen de las actividades
identificadas a partir de reuniones y la información ingresada
por el colaborador.

{resumen_actividades}

Instrucciones:
- Resume la información de forma clara y ordenada
- No combines el Volumen con la Unidad de Medida,
  dalos por separado ya que el volumen es respecto a la Frecuencia
- Usa lenguaje sencillo y no técnico
- No agregues información nueva
- No hagas cálculos adicionales
- No corrijas ortografía a menos que sean la misma palabra en distinta capitalización
- Finaliza preguntando si la información es correcta o si desea hacer ajustes
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 8. LECTURA DE ARCHIVO VTT DESDE FILEUPLOAD (JUPYTER)
    # -----------------------------------------------------------------------
    def obtener_contenido_vtt(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo")
        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]
        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes().decode("utf-8")
        elif isinstance(contenido_raw, bytes):
            return contenido_raw.decode("utf-8")
        else:
            raise TypeError("Tipo de contenido no soportado")

    # -----------------------------------------------------------------------
    # 8b. FUSIÓN DE ACTIVIDADES DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario(
        self, actividades_base, actividades_enriquecidas, inputs_usuario
    ):
        actividades_finales = []
        for base, ia, user in zip(
            actividades_base, actividades_enriquecidas, inputs_usuario
        ):
            actividades_finales.append({
                "nombre": ia["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": user["frecuencia"],
                "volumen": user["volumen"],
                "proceso_area": user["proceso_area"],
                "autonomia": user["autonomia"],
                "observaciones": user.get("observaciones", "")
            })
        return actividades_finales

    # -----------------------------------------------------------------------
    # 9. CONFIRMAR LA INFORMACIÓN POR PARTE DEL USUARIO
    # -----------------------------------------------------------------------
    def confirmar_informacion(self, resumen_texto):
        print("\n" + "═" * 60)
        print("  📋  RESUMEN FINAL DEL LEVANTAMIENTO")
        print("═" * 60)
        print(resumen_texto)
        print("═" * 60)
        print(
            "\n  ℹ️  Este es el resumen definitivo. Si detectas un error,\n"
            "     comunícalo al encargado del proceso."
        )
        while True:
            respuesta = input(
                "\n¿Confirmas que la información es correcta? (si / no): "
            ).strip().lower()
            if respuesta in ("si", "no"):
                if respuesta == "no":
                    print(
                        "\n  ℹ️  El levantamiento se registrará con la información\n"
                        "     ingresada. Las correcciones deben solicitarse al\n"
                        "     encargado del proceso."
                    )
                return True
            print("  ⚠️ Por favor escribe 'si' o 'no'.")

    # -----------------------------------------------------------------------
    # 10. CALCULAR MÉTRICAS
    # -----------------------------------------------------------------------
    def calcular_metricas(self, actividades):
        DIAS_LABORALES_MES = 21
        MINUTOS_JORNADA_MES = 220 * 60
        JORNADA_LABORAL_DIARIA = 8.5

        FACTOR_FRECUENCIA = {
            "Diario": 1,
            "Semanal": 1 / 5,
            "Quincenal": 1 / 10,
            "Mensual": 1 / 21,
            "Bimensual": 1 / 42,
            "Trimestral": 1 / 63,
            "Semestral": 1 / 126,
            "Anual": 1 / 252
        }

        carga_w_por_phva = {}
        cantidad_por_phva = {}
        carga_w_por_frecuencia = {}
        cantidad_por_frecuencia = {}
        carga_w_por_proceso_area = {}
        cantidad_por_proceso_area = {}

        total_carga_w_sin_tm = 0
        total_carga_trabajo_individual = 0
        total_minutos_diarios = 0

        for act in actividades:
            tiempo_base = act["duracion_min"]
            volumen = act["volumen"]
            frecuencia = act["frecuencia"]
            autonomia = act["autonomia"] / 100
            phva = act.get("phva", "SIN CLASIFICAR")
            proceso_area = act.get("proceso_area", "SIN CLASIFICAR")

            factor = FACTOR_FRECUENCIA.get(frecuencia, 0)
            minutos_diarios = tiempo_base * volumen * factor
            minutos_mes = minutos_diarios * DIAS_LABORALES_MES
            carga_w_sin_tm = minutos_mes / MINUTOS_JORNADA_MES
            carga_trabajo_individual = carga_w_sin_tm * autonomia

            act["metricas"] = {
                "minutos_diarios": round(minutos_diarios, 2),
                "minutos_mes": round(minutos_mes, 2),
                "carga_w_sin_tm": round(carga_w_sin_tm, 4),
                "carga_trabajo_individual": round(carga_trabajo_individual, 4)
            }

            total_carga_w_sin_tm += carga_w_sin_tm
            total_carga_trabajo_individual += carga_trabajo_individual
            total_minutos_diarios += minutos_diarios

            carga_w_por_phva[phva] = carga_w_por_phva.get(phva, 0) + carga_w_sin_tm
            cantidad_por_phva[phva] = cantidad_por_phva.get(phva, 0) + 1
            carga_w_por_frecuencia[frecuencia] = (
                carga_w_por_frecuencia.get(frecuencia, 0) + carga_w_sin_tm
            )
            cantidad_por_frecuencia[frecuencia] = (
                cantidad_por_frecuencia.get(frecuencia, 0) + 1
            )
            carga_w_por_proceso_area[proceso_area] = (
                carga_w_por_proceso_area.get(proceso_area, 0) + carga_w_sin_tm
            )
            cantidad_por_proceso_area[proceso_area] = (
                cantidad_por_proceso_area.get(proceso_area, 0) + 1
            )

        almuerzo = 1 / 8
        baño_agua_etc = (3 * 5) / 60
        break_15min = 15 / 60
        latencia_software = 10 / 60
        cliente_ext_o_int = 20 / 60
        tiempo_muerto = sum([
            almuerzo / JORNADA_LABORAL_DIARIA,
            baño_agua_etc / JORNADA_LABORAL_DIARIA,
            break_15min / JORNADA_LABORAL_DIARIA,
            latencia_software / JORNADA_LABORAL_DIARIA,
            cliente_ext_o_int / JORNADA_LABORAL_DIARIA
        ])
        factor_tiempo_neto = round(1 - tiempo_muerto, 4)

        def porcentajes(dic):
            total = sum(dic.values())
            return {
                k: round(v / total, 4) if total > 0 else 0
                for k, v in dic.items()
            }

        horas_diarias_requeridas = total_minutos_diarios / 60
        horas_netas_por_persona = JORNADA_LABORAL_DIARIA * factor_tiempo_neto

        return {
            "carga_trabajo_phva": {k: round(v, 4) for k, v in carga_w_por_phva.items()},
            "cantidad_actividades_phva": cantidad_por_phva,
            "porcentaje_actividades_phva": porcentajes(carga_w_por_phva),
            "carga_trabajo_frecuencia": {k: round(v, 4) for k, v in carga_w_por_frecuencia.items()},
            "cantidad_actividades_frecuencia": cantidad_por_frecuencia,
            "porcentaje_actividades_frecuencia": porcentajes(carga_w_por_frecuencia),
            "carga_trabajo_proceso_area": {
                k: round(v, 4) for k, v in carga_w_por_proceso_area.items()
            },
            "cantidad_actividades_proceso_area": cantidad_por_proceso_area,
            "porcentaje_actividades_proceso_area": porcentajes(carga_w_por_proceso_area),
            "total_carga_w_sin_tm": round(total_carga_w_sin_tm, 4),
            "total_carga_trabajo_individual": round(total_carga_trabajo_individual, 4),
            "minutos_diarios_empleados": round(total_minutos_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
            "jornada_laboral_diaria": JORNADA_LABORAL_DIARIA,
            "factor_tiempo_neto_productivo": factor_tiempo_neto,
            "horas_netas_efectivas_por_persona": round(horas_netas_por_persona, 2),
            "numero_personas_requeridas": round(
                (total_minutos_diarios * (1 + tiempo_muerto) / 60) / horas_netas_por_persona, 3
            ),
            "tiempo_muerto": round(tiempo_muerto, 4),
            "horas_diarias_requeridas_final": round(
                total_minutos_diarios * (1 + tiempo_muerto) / 60, 2
            )
        }

    # -----------------------------------------------------------------------
    # 11. ANÁLISIS IA PARA EL ANALISTA
    # -----------------------------------------------------------------------
    def analisis_analista_ia(self, actividades, metricas, contexto):
        prompt = f"""
Eres un ANALISTA SENIOR en Levantamiento de Cargas Laborales y Dimensionamiento Operativo.
Tu tarea NO es recalcular datos, sino INTERPRETAR, VALIDAR COHERENCIA y EMITIR JUICIO PROFESIONAL
a partir de la información suministrada.

=========================
CONTEXTO ORGANIZACIONAL
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES ANALIZADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

=========================
MÉTRICAS CALCULADAS
=========================
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES DE ANÁLISIS
=========================

1. Analiza el NIVEL DE CARGA LABORAL.
2. Evalúa la COHERENCIA del resultado.
3. Interpreta el BALANCE PHVA.
4. Identifica RIESGOS OPERATIVOS.
5. Detecta OPORTUNIDADES DE AUTOMATIZACIÓN O MEJORA.
6. Formula RECOMENDACIONES EJECUTIVAS.

Menciona los valores numéricos mientras das el análisis e interpreta su significado.

Responde en español con lenguaje profesional, estructurado en los bloques anteriores.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 12. CAPTURAR CARGO Y VICEPRESIDENCIA (WIDGETS — flujo Jupyter)
    # -----------------------------------------------------------------------
    def capturar_contexto_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        cargo = widgets.Text(
            description="Cargo:",
            placeholder="Ej: Analista de Datos",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        vicepresidencia = widgets.Text(
            description="Vicepresidencia:",
            placeholder="Ej: Tecnología",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        display(cargo, vicepresidencia)
        return {"cargo": cargo, "vicepresidencia": vicepresidencia}

    # -----------------------------------------------------------------------
    # 13. OBTENER VALORES DE CONTEXTO
    # -----------------------------------------------------------------------
    def obtener_contexto_valores(self, contexto_widgets):
        return {
            "cargo": contexto_widgets["cargo"].value,
            "vicepresidencia": contexto_widgets["vicepresidencia"].value
        }

    # -----------------------------------------------------------------------
    # 14–15. CAPTURA DE INPUTS POR ACTIVIDAD (WIDGETS)
    # -----------------------------------------------------------------------
    def capturar_inputs_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        frecuencia = widgets.Dropdown(
            options=self.FRECUENCIAS_VALIDAS,
            description="Frecuencia:",
            style={'description_width': '120px'}
        )
        volumen = widgets.IntText(
            description="Volumen:",
            value=1,
            style={'description_width': '120px'}
        )
        autonomia = widgets.IntSlider(
            description="Autonomía (%):",
            min=0, max=100, value=80,
            style={'description_width': '120px'}
        )
        proceso_area = widgets.Textarea(
            description="Proceso del área:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        observaciones = widgets.Textarea(
            description="Observaciones:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        display(frecuencia, volumen, autonomia, proceso_area, observaciones)
        return {
            "frecuencia": frecuencia,
            "volumen": volumen,
            "proceso_area": proceso_area,
            "autonomia": autonomia,
            "observaciones": observaciones
        }

    def obtener_actividades_unicas(self, actividades):
        vistas = set()
        unicas = []
        for act in actividades:
            nombre = act["actividad"].strip().lower()
            if nombre not in vistas:
                vistas.add(nombre)
                unicas.append(act)
        return unicas

    def capturar_inputs_usuario_actividades_unicas(self, actividades):
        actividades_unicas = self.obtener_actividades_unicas(actividades)
        inputs_por_actividad = {}
        for act in actividades_unicas:
            print(f"\n Actividad: {act['actividad']}")
            inputs_por_actividad[act["actividad"].strip().lower()] = (
                self.capturar_inputs_usuario()
            )
        return inputs_por_actividad

    def obtener_inputs_usuario_valores(self, inputs_widgets):
        return {
            "frecuencia": inputs_widgets["frecuencia"].value,
            "volumen": inputs_widgets["volumen"].value,
            "autonomia": inputs_widgets["autonomia"].value,
            "proceso_area": inputs_widgets["proceso_area"].value,
            "observaciones": inputs_widgets["observaciones"].value
        }

    # -----------------------------------------------------------------------
    # 16. EDITAR ACTIVIDADES (GENERAL)
    # -----------------------------------------------------------------------
    def editar_actividades(self, actividades):
        while True:
            if not actividades:
                print("\n⚠️ No hay actividades para editar.")
                break

            print("\n✏️ ACTIVIDADES DISPONIBLES:")
            for i, act in enumerate(actividades, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nIngrese el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades)):
                print("⚠️ Opción inválida")
                continue

            idx = int(opcion) - 1
            actividad = actividades[idx]

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                actividad_original = copy.deepcopy(actividad)
                confirmacion = input(
                    f"⚠️ ¿Seguro que deseas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                    confirmar_final = input(
                        "¿Confirmas la eliminación? (si / no): "
                    ).strip().lower()
                    if confirmar_final != "si":
                        actividades.insert(idx, actividad_original)
                        print("↩️ Eliminación revertida.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividad[campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo modificado: {campo}")
            print(f"- Valor anterior: {actividad_original.get(campo)}")
            print(f"- Nuevo valor: {nuevo_valor}")

            confirmar_cambio = input(
                "\n¿Confirmas este cambio? (si / no): "
            ).strip().lower()
            if confirmar_cambio != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 16b. EDITAR UNA ACTIVIDAD INDIVIDUAL — selección de campo por NÚMERO
    # -----------------------------------------------------------------------
    def _editar_actividad_individual(self, actividad):
        """
        Permite editar campos individuales de UNA actividad específica
        sin tener que reingresar toda la información.

        CAMBIO: el usuario selecciona el campo por número, no escribiendo
        el nombre de la columna, lo que evita errores tipográficos y
        hace el flujo más ágil.
        """
        # Lista ordenada de (nombre_campo, etiqueta_legible)
        CAMPOS = [
            ("nombre",       "Nombre de la actividad"),
            ("frecuencia",   "Frecuencia"),
            ("volumen",      "Volumen"),
            ("duracion_min", "Duración (minutos)"),
            ("autonomia",    "Autonomía (%)"),
            ("proceso_area", "Proceso del área"),
            ("observaciones","Observaciones"),
        ]

        print("\n" + "─" * 60)
        print("  ✏️  EDICIÓN DE CAMPOS — Actividad")
        print("─" * 60)

        while True:
            # Mostrar menú numerado con valor actual
            print("\nCampos disponibles para editar:")
            for idx, (campo, etiqueta) in enumerate(CAMPOS, start=1):
                valor_actual = actividad.get(campo, "N/A")
                print(f"  {idx}) {etiqueta:<30} → {valor_actual}")
            print(f"  {len(CAMPOS) + 1}) Listo (terminar edición)")

            resp = input(
                f"\nEscribe el número del campo a modificar "
                f"(1-{len(CAMPOS)}) o {len(CAMPOS) + 1} para terminar: "
            ).strip()

            # Validar que sea un número dentro del rango
            if not resp.isdigit():
                print("⚠️ Ingresa solo el número de la opción.")
                continue

            opcion = int(resp)

            if opcion == len(CAMPOS) + 1:
                break

            if not (1 <= opcion <= len(CAMPOS)):
                print(f"⚠️ Elige un número del 1 al {len(CAMPOS) + 1}.")
                continue

            campo, etiqueta = CAMPOS[opcion - 1]

            # ── Captura del nuevo valor según el campo ────────────────────
            if campo == "frecuencia":
                opciones_str = "  ".join(
                    f"{i+1}) {f}" for i, f in enumerate(self.FRECUENCIAS_VALIDAS)
                )
                print(f"\n  Frecuencias válidas:\n  {opciones_str}")
                while True:
                    r = input("Nueva frecuencia (número u opción): ").strip()
                    if r.isdigit() and 1 <= int(r) <= len(self.FRECUENCIAS_VALIDAS):
                        nuevo_valor = self.FRECUENCIAS_VALIDAS[int(r) - 1]
                        break
                    elif r.capitalize() in self.FRECUENCIAS_VALIDAS:
                        nuevo_valor = r.capitalize()
                        break
                    else:
                        print(
                            f"⚠️ Opción no válida. Elige un número del "
                            f"1 al {len(self.FRECUENCIAS_VALIDAS)}."
                        )

            elif campo in ("volumen", "duracion_min"):
                while True:
                    r = input(f"Nuevo valor para '{etiqueta}' (entero > 0): ").strip()
                    if r.isdigit() and int(r) > 0:
                        nuevo_valor = int(r)
                        break
                    print("⚠️ Ingresa un número entero mayor a 0.")

            elif campo == "autonomia":
                autonomia_opciones = list(range(5, 105, 5))
                fila = "  " + "  ".join(f"{p}%" for p in autonomia_opciones)
                print(f"\n  Opciones de autonomía:\n{fila}")
                while True:
                    r = input("Nueva autonomía (%): ").strip().replace("%", "").strip()
                    if r.isdigit() and int(r) in autonomia_opciones:
                        nuevo_valor = int(r)
                        break
                    print("⚠️ Elige un valor de 5 en 5 entre 5% y 100%.")

            else:
                nuevo_valor = input(f"Nuevo valor para '{etiqueta}': ").strip()

            # ── Confirmar el cambio ───────────────────────────────────────
            valor_anterior = actividad.get(campo)
            actividad[campo] = nuevo_valor

            print(f"\n🧾 Cambio aplicado:")
            print(f"  Campo     : {etiqueta}")
            print(f"  Antes     : {valor_anterior}")
            print(f"  Después   : {nuevo_valor}")

            confirmar = input("¿Confirmas este cambio? (si / no): ").strip().lower()
            if confirmar != "si":
                actividad[campo] = valor_anterior
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividad

    # -----------------------------------------------------------------------
    # 17. FUSIÓN ACTIVIDADES NO DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario_nodiarias(
        self, actividades_base, actividades_enriquecidas
    ):
        actividades_finales_nodiarias = []
        for base, ia in zip(actividades_base, actividades_enriquecidas):
            actividades_finales_nodiarias.append({
                "nombre": base["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": base["frecuencia"],
                "volumen": base["volumen"],
                "proceso_area": base["proceso_area"],
                "autonomia": base["autonomia"],
                "observaciones": base.get("observaciones", "")
            })
        return actividades_finales_nodiarias

    # -----------------------------------------------------------------------
    # 18. VALIDACIÓN DE COHERENCIA GLOBAL
    # -----------------------------------------------------------------------
    def validar_coherencia_global(self, actividades, contexto):
        prompt = f"""
Actúa como un validador técnico de coherencia de actividades laborales.

El volumen representa el número de veces que se ejecuta la actividad
dentro del periodo definido por la frecuencia.

Tu tarea:
1. Detectar únicamente incoherencias internas.
2. No hacer juicios del rol ni usar información externa.

Debes validar únicamente:
- volumen × duración dentro del periodo de la frecuencia
- acumulaciones problemáticas dentro del mismo periodo
- duraciones extremadamente bajas o altas en relación con el volumen
- inconsistencias temporales si existen

⚠️ IMPORTANTE:
- Solo muestra las actividades donde detectes incoherencias.
- NO muestres actividades que estén correctas.

Formato de salida obligatorio:

------------------------------------------------------------
🔎 VALIDACIÓN DE COHERENCIA

Para cada actividad con incoherencia:

📌 Actividad: [Nombre exacto]

Campos evaluados:
- Frecuencia: [valor]
- Volumen: [valor]
- Duración por unidad: [valor + unidad]

Cálculo implícito:
[volumen × duración = total dentro del periodo]

Incoherencia detectada:
- [Descripción técnica breve]

------------------------------------------------------------

🧾 Resumen general:
- Total de actividades analizadas: [número]
- Actividades con incoherencias: [número]
- Errores matemáticos explícitos: [sí/no]

Al final devuelve SIEMPRE un bloque JSON así:

{{
    "actividades_incoherentes": ["Nombre 1", "Nombre 2"]
}}

Si no hay incoherencias:

{{
    "actividades_incoherentes": []
}}

=========================
CONTEXTO
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES REGISTRADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

Responde en español.
"""
        reply = llm.invoke(prompt)
        content = reply.content
        actividades_incoherentes = []
        try:
            json_start = content.rfind("{")
            if json_start != -1:
                json_text = content[json_start:]
                json_data = json.loads(json_text)
                actividades_incoherentes = json_data.get(
                    "actividades_incoherentes", []
                )
        except Exception:
            actividades_incoherentes = []
        return content, actividades_incoherentes

    # -----------------------------------------------------------------------
    # 18.1 EDITAR ACTIVIDADES INCOHERENTES
    # -----------------------------------------------------------------------
    def editar_actividades_incoherentes(self, actividades, actividades_incoherentes):
        actividades_filtradas = [
            act for act in actividades
            if act["nombre"] in actividades_incoherentes
        ]
        if not actividades_filtradas:
            print("\n✅ No hay actividades con incoherencias para editar.")
            return actividades

        while True:
            print("\n⚠️ ACTIVIDADES CON INCOHERENCIAS:")
            for i, act in enumerate(actividades_filtradas, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nSeleccione el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (
                1 <= int(opcion) <= len(actividades_filtradas)
            ):
                print("⚠️ Opción inválida")
                continue

            actividad = actividades_filtradas[int(opcion) - 1]
            idx = next(
                i for i, a in enumerate(actividades)
                if a["nombre"] == actividad["nombre"]
            )

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                confirmacion = input(
                    f"¿Confirmas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividades[idx][campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo: {campo}")
            print(f"- Antes: {actividad_original.get(campo)}")
            print(f"- Después: {nuevo_valor}")

            confirmar = input("\n¿Confirmas el cambio? (si / no): ").strip().lower()
            if confirmar != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 19. RESUMEN AUTOMÁTICO DE CAMBIOS
    # -----------------------------------------------------------------------
    def generar_resumen_cambios(self, antes, despues):
        prompt = f"""
Eres un asistente que compara versiones de información.

REGLAS:
- No recalcules métricas.
- No hagas análisis.
- Solo describe qué cambió.
- Sé breve y claro.
- Si no hubo cambios, indícalo.

====================
ANTES
====================
{json.dumps(antes, indent=2, ensure_ascii=False)}

====================
DESPUÉS
====================
{json.dumps(despues, indent=2, ensure_ascii=False)}

Responde en español en formato claro y organizado.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # =======================================================================
    # ★ FLUJO CONVERSACIONAL PARA ACTIVIDADES NO DIARIAS
    # =======================================================================

    # -----------------------------------------------------------------------
    # A. VALIDAR UNA ACTIVIDAD CON IA (coherencia individual)
    # -----------------------------------------------------------------------
    def _validar_coherencia_actividad_ia(self, actividad, min_disponibles=None):
        tope_diario = round(min_disponibles, 1) if min_disponibles is not None else 510
        tope_label  = (
            f"{tope_diario} min disponibles en la jornada "
            f"(descontando actividades ya registradas)"
            if min_disponibles is not None
            else "510 min (8.5 h, jornada completa)"
        )

        prompt = f"""
Actúa como validador técnico de coherencia de actividades laborales.

Tienes UNA actividad con los siguientes datos:
{json.dumps(actividad, indent=2, ensure_ascii=False)}

Definición clave:
- "volumen" = número de veces que se ejecuta la actividad dentro del
  periodo definido por "frecuencia".
- "duracion_min" = minutos que tarda UNA ejecución.

Tiempo disponible en la jornada diaria para validar: {tope_label}

Debes verificar:
1. ¿El volumen × duración_min supera el tiempo disponible en el periodo?
   Usa el tiempo disponible indicado arriba como tope para frecuencia Diaria.
   Para otras frecuencias multiplica por los días del periodo:
   - Semanal: {tope_diario} × 5
   - Quincenal: {tope_diario} × 10
   - Mensual: {tope_diario} × 21
   - Bimensual: {tope_diario} × 42
   - Trimestral: {tope_diario} × 63
   - Semestral: {tope_diario} × 126
   - Anual: {tope_diario} × 252
2. ¿La duración individual parece extremadamente corta (< 1 min) o
   larga (> 480 min) para el tipo de actividad?
3. ¿Hay alguna otra inconsistencia lógica evidente?

Si hay incoherencia, genera recomendaciones CONCRETAS y ACCIONABLES sobre
qué valores específicos cambiar (campo, valor actual → valor sugerido).

Responde EXCLUSIVAMENTE con un JSON válido, sin texto adicional:

{{
  "hay_incoherencia": true | false,
  "explicacion": "descripción breve y clara de la incoherencia, o cadena vacía si no hay",
  "recomendaciones": [
    "Recomendación concreta 1 (campo: valor actual → valor sugerido)",
    "Recomendación concreta 2 (si aplica)"
  ]
}}

Si no hay incoherencia, devuelve:
{{
  "hay_incoherencia": false,
  "explicacion": "",
  "recomendaciones": []
}}
"""
        reply = llm.invoke(prompt)
        try:
            resultado = self._parse_json_seguro(reply.content)
            return (
                resultado.get("hay_incoherencia", False),
                resultado.get("explicacion", ""),
                resultado.get("recomendaciones", []),
            )
        except Exception:
            return False, "", []

    # -----------------------------------------------------------------------
    # B. MOSTRAR INCOHERENCIA Y APLICAR CORRECCIONES AUTOMÁTICAMENTE
    # -----------------------------------------------------------------------
    def _mostrar_incoherencia_y_corregir(
        self, actividad, explicacion, recomendaciones, min_disponibles=None
    ):
        def _parsear_y_aplicar(act, recs):
            import re as _re
            campos_agente = {"volumen", "duracion_min"}
            cambios = []

            for rec in recs:
                m = _re.search(
                    r"([a-zA-Z_]+)\s*:\s*[\w\s,.\[\]]+?→\s*([\w.]+)",
                    rec
                )
                if not m:
                    continue
                campo   = m.group(1).strip().lower()
                val_str = m.group(2).strip()

                if campo not in campos_agente:
                    continue

                try:
                    nuevo_valor = int(float(val_str))
                    if nuevo_valor <= 0:
                        continue
                except ValueError:
                    continue

                valor_anterior = act.get(campo)
                act[campo] = nuevo_valor
                cambios.append((campo, valor_anterior, nuevo_valor))

            return act, cambios

        expl_actual = explicacion
        recs_actual = recomendaciones
        iteracion   = 0

        while True:
            print("\n" + "─" * 60)
            print("⚠️  INCOHERENCIA DETECTADA — corrigiendo automáticamente")
            print("─" * 60)
            print(f"   {expl_actual}")

            if recs_actual:
                print("\n💡 Correcciones del agente:")
                for i, rec in enumerate(recs_actual, start=1):
                    print(f"   {i}. {rec}")
            print("─" * 60)

            actividad, cambios = _parsear_y_aplicar(actividad, recs_actual)

            if cambios:
                print("\n🔧 Cambios aplicados automáticamente:")
                for campo, antes, despues in cambios:
                    print(f"   • {campo}: {antes} → {despues}")
            else:
                print(
                    "\nℹ️  Las correcciones involucran campos que solo el usuario\n"
                    "   conoce (frecuencia, proceso del área). Se conservan los\n"
                    "   valores ingresados y se continuará con la validación final."
                )
                break

            print("\n🔍 Re-validando coherencia...")
            hay_inc, expl_nuevo, recs_nuevo = self._validar_coherencia_actividad_ia(
                actividad, min_disponibles=min_disponibles
            )

            if not hay_inc:
                print("✅ Datos corregidos y coherentes.")
                break

            iteracion += 1
            if iteracion >= 3:
                print(
                    "ℹ️  Se alcanzó el límite de correcciones automáticas.\n"
                    "   Se continúa con los valores actuales."
                )
                break

            expl_actual = expl_nuevo
            recs_actual = recs_nuevo

        return actividad

    # -----------------------------------------------------------------------
    # C1. CALCULAR MÉTRICAS PARCIALES ACUMULADAS
    # -----------------------------------------------------------------------
    def _calcular_metricas_parciales(self, actividades_hasta_ahora):
        DIAS_LABORALES_MES   = 21
        MINUTOS_JORNADA_MES  = 220 * 60
        JORNADA_DIARIA       = 8.5

        FACTOR_FRECUENCIA = {
            "Diario":     1,
            "Semanal":    1 / 5,
            "Quincenal":  1 / 10,
            "Mensual":    1 / 21,
            "Bimensual":  1 / 42,
            "Trimestral": 1 / 63,
            "Semestral":  1 / 126,
            "Anual":      1 / 252,
        }

        tiempo_muerto = sum([
            (1 / 8)        / JORNADA_DIARIA,
            ((3 * 5) / 60) / JORNADA_DIARIA,
            (15 / 60)      / JORNADA_DIARIA,
            (10 / 60)      / JORNADA_DIARIA,
            (20 / 60)      / JORNADA_DIARIA,
        ])
        factor_neto = 1 - tiempo_muerto
        horas_netas_por_persona = JORNADA_DIARIA * factor_neto

        total_min_diarios      = 0.0
        total_carga_w_sin_tm   = 0.0
        total_carga_individual = 0.0
        metricas_por_actividad = []

        for act in actividades_hasta_ahora:
            factor    = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            min_dia   = act["duracion_min"] * act["volumen"] * factor
            min_mes   = min_dia * DIAS_LABORALES_MES
            carga_w   = min_mes / MINUTOS_JORNADA_MES
            carga_ind = carga_w * (act["autonomia"] / 100)

            total_min_diarios      += min_dia
            total_carga_w_sin_tm   += carga_w
            total_carga_individual += carga_ind

            metricas_por_actividad.append({
                "nombre":           act["nombre"],
                "frecuencia":       act.get("frecuencia"),
                "min_diarios":      round(min_dia,  2),
                "carga_w_sin_tm":   round(carga_w,  4),
                "carga_individual": round(carga_ind, 4),
            })

        horas_diarias_requeridas = total_min_diarios / 60
        horas_con_tm = total_min_diarios * (1 + tiempo_muerto) / 60
        personas_req = horas_con_tm / horas_netas_por_persona if horas_netas_por_persona else 0

        return {
            "total_min_diarios":        round(total_min_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
            "horas_con_tiempo_muerto":  round(horas_con_tm, 2),
            "total_carga_w_sin_tm":     round(total_carga_w_sin_tm, 4),
            "total_carga_individual":   round(total_carga_individual, 4),
            "personas_requeridas":      round(personas_req, 3),
            "jornada_laboral_diaria":   JORNADA_DIARIA,
            "horas_netas_por_persona":  round(horas_netas_por_persona, 2),
            "factor_tiempo_neto":       round(factor_neto, 4),
            "num_actividades":          len(actividades_hasta_ahora),
            "metricas_por_actividad":   metricas_por_actividad,
        }

    # -----------------------------------------------------------------------
    # C2. ANÁLISIS CONTEXTUAL POR ACTIVIDAD (LLM)
    # -----------------------------------------------------------------------
    def _analisis_actividad_en_contexto(
        self, actividad_nueva, actividades_previas, metricas, contexto
    ):
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.
Tu rol en este momento es acompañar al colaborador durante el registro
de sus actividades y darle retroalimentación útil tras registrar cada una.

=========================
CONTEXTO DEL COLABORADOR
=========================
Cargo           : {contexto.get("cargo")}
Vicepresidencia : {contexto.get("vicepresidencia")}

=========================
ACTIVIDAD RECIÉN REGISTRADA
=========================
{json.dumps(actividad_nueva, indent=2, ensure_ascii=False)}

=========================
ACTIVIDADES YA REGISTRADAS ANTES DE ESTA
=========================
{json.dumps(actividades_previas, indent=2, ensure_ascii=False) if actividades_previas else "Ninguna (esta es la primera actividad registrada)"}

=========================
MÉTRICAS INTERNAS CALCULADAS (para tu análisis — NO las expongas)
=========================
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES ESTRICTAS
=========================
1. Usa las métricas internas SOLO como base de tu interpretación.
   NO menciones ni repitas los valores numéricos de carga laboral,
   dotación, minutos diarios ni factores de tiempo al usuario.

2. Tu análisis debe cubrir EXACTAMENTE estos dos puntos:
   a) ¿Qué representa esta actividad para la carga del colaborador?
   b) ¿Cómo se ve la carga acumulada hasta ahora en relación con
      una jornada laboral normal?

3. Si es la primera actividad, solo analiza el punto (a).

4. Cierra con una frase breve y motivadora que invite al colaborador
   a continuar con el registro.

5. NO hagas preguntas. NO solicites correcciones.

Responde en español, tono profesional pero cercano.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # C. MOSTRAR RESUMEN DE UNA ACTIVIDAD Y PEDIR CONFIRMACIÓN
    # -----------------------------------------------------------------------
    def _confirmar_actividad(self, actividad, numero, actividades_previas, contexto):
        todas    = actividades_previas + [actividad]
        metricas = self._calcular_metricas_parciales(todas)

        analisis_interno = self._analisis_actividad_en_contexto(
            actividad_nueva     = actividad,
            actividades_previas = actividades_previas,
            metricas            = metricas,
            contexto            = contexto,
        )
        actividad["_analisis_interno"] = analisis_interno

        while True:
            print("\n" + "═" * 60)
            print(f"  RESUMEN — Actividad {numero}")
            print("═" * 60)
            print(f"  Nombre            : {actividad.get('nombre')}")
            if actividad.get("descripcion"):
                print(f"  Descripción       : {actividad.get('descripcion')}")
            print(f"  Frecuencia        : {actividad.get('frecuencia')}")
            print(f"  Volumen           : {actividad.get('volumen')} "
                  f"vez/veces por periodo")
            print(f"  Duración          : {actividad.get('duracion_min')} min "
                  f"por ejecución")
            print(f"  Autonomía         : {actividad.get('autonomia')}%")
            print(f"  Proceso del área  : {actividad.get('proceso_area') or '—'}")
            if actividad.get("observaciones"):
                print(f"  Observaciones     : {actividad.get('observaciones')}")
            print("═" * 60)

            resp = input(
                "\n¿Esta información es correcta? (si / no): "
            ).strip().lower()

            if resp == "si":
                return True

            print(
                "\n🔄 Puedes modificar los campos que necesites "
                "sin tener que empezar de cero."
            )
            actividad = self._editar_actividad_individual(actividad)
            print("\n✅ Cambios aplicados. Revisando el resumen actualizado...")

    # -----------------------------------------------------------------------
    # D. RECOGER UNA ACTIVIDAD COMPLETA — incluye selección de proceso del área
    # -----------------------------------------------------------------------
    # -----------------------------------------------------------------------
    # D.0  VERIFICAR PROCESO DEL ÁREA Y OBSERVACIONES CON IA
    # -----------------------------------------------------------------------
    def _verificar_proceso_y_observaciones(
        self, actividad: dict, contexto: dict
    ) -> dict:
        proceso_original      = actividad.get("proceso_area", "").strip()
        observaciones_original = actividad.get("observaciones", "").strip()

        prompt = f"""
Eres un corrector ortográfico especializado en textos laborales en español.

Tu única tarea es revisar los campos "proceso_area" y "observaciones" e
identificar errores ortográficos, de acentuación o de capitalización.

REGLAS ESTRICTAS:
1. PROCESO DEL ÁREA:
   - SOLO corrige errores ortográficos (tildes, mayúsculas iniciales,
     errores de escritura evidentes como "getsion" → "gestión").
   - NO cambies el contenido, el significado ni las palabras elegidas
     por el usuario. El texto pertenece al usuario y debes respetarlo.
   - NO amplíes, NO hagas más específico, NO reescribas con otras palabras.
   - Si no hay errores ortográficos → devuelve exactamente el texto original.

2. OBSERVACIONES (pueden estar vacías):
   - Si están vacías → son válidas, no cambies nada.
   - Si tienen contenido, verifica:
     a) ¿Son coherentes con la actividad? ¿Aportan información nueva?
     b) Si son redundantes (repiten nombre/frecuencia/volumen ya capturados)
        → sugiere dejarlas vacías o propón una versión mejorada.
     c) Si contienen información relevante y útil → márcalas como válidas.

Datos a revisar:
- Proceso del área : "{proceso_original}"
- Observaciones    : "{observaciones_original}"

Responde EXCLUSIVAMENTE con un JSON válido:

{{
  "proceso_corregido": "texto con ortografía corregida (o idéntico al original si no hay errores)",
  "hubo_correccion_ortografica": true | false,
  "observaciones_validas": true | false,
  "observaciones_sugeridas": "texto mejorado, cadena vacía si deben eliminarse, o igual si ya son válidas",
  "razon_observaciones": "explicación breve o cadena vacía"
}}
"""
        try:
            reply = llm.invoke(prompt)
            resultado = self._parse_json_seguro(reply.content)
        except Exception:
            return actividad

        cambios = []

        # ── Proceso del área: solo corrección ortográfica ─────────────────
        proceso_corregido = resultado.get("proceso_corregido", proceso_original).strip()
        hubo_correccion   = resultado.get("hubo_correccion_ortografica", False)
        if hubo_correccion and proceso_corregido and proceso_corregido != proceso_original:
            actividad["proceso_area"] = proceso_corregido
            cambios.append((
                "Proceso del área",
                proceso_original or "(vacío)",
                proceso_corregido,
                "Corrección ortográfica aplicada"
            ))

        if observaciones_original and not resultado.get("observaciones_validas", True):
            obs_sugerida = resultado.get("observaciones_sugeridas", "").strip()
            if obs_sugerida.lower() != observaciones_original.lower():
                actividad["observaciones"] = obs_sugerida
                cambios.append((
                    "Observaciones",
                    observaciones_original,
                    obs_sugerida or "(eliminadas)",
                    resultado.get("razon_observaciones", "")
                ))

        if cambios:
            print("\n" + "─" * 60)
            print("📝  AJUSTE AUTOMÁTICO — Proceso del área / Observaciones")
            print("─" * 60)
            for campo, antes, despues, razon in cambios:
                print(f"\n  Campo   : {campo}")
                if razon:
                    print(f"  Razón   : {razon}")
                print(f"  Antes   : \"{antes}\"")
                print(f"  Después : \"{despues}\"")
            print("─" * 60)

        return actividad

    def _preguntar_proceso_area(self, procesos_registrados):
        procesos_unicos = list(dict.fromkeys(
            p for p in procesos_registrados if p
        ))

        if procesos_unicos:
            print("\n¿A qué proceso del área pertenece esta actividad?")
            print("  Procesos ya registrados:")
            for i, proc in enumerate(procesos_unicos, start=1):
                print(f"    {i}) {proc}")
            print(f"    {len(procesos_unicos) + 1}) Ingresar un proceso nuevo")

            while True:
                resp = input(
                    f"Selecciona un número del 1 al "
                    f"{len(procesos_unicos) + 1}: "
                ).strip()

                if resp.isdigit():
                    opcion = int(resp)
                    if 1 <= opcion <= len(procesos_unicos):
                        return procesos_unicos[opcion - 1]
                    elif opcion == len(procesos_unicos) + 1:
                        break
                    else:
                        print(
                            f"⚠️ Elige un número del 1 al "
                            f"{len(procesos_unicos) + 1}."
                        )
                else:
                    print("⚠️ Por favor, ingresa un número.")

        proceso = ""
        while not proceso:
            proceso = input(
                "\n¿A qué proceso del área pertenece esta actividad? "
                "(Ej: Gestión de proveedores, Reportes, Atención al cliente): "
            ).strip()
            if not proceso:
                print("⚠️ Por favor, ingresa el proceso del área.")
        return proceso

    def _preguntar_actividad(self, numero, procesos_registrados=None, contexto=None):
        if procesos_registrados is None:
            procesos_registrados = []

        print("\n" + "─" * 60)
        print(f"  📋  ACTIVIDAD {numero}")
        print("─" * 60)

        nombre = ""
        while not nombre:
            nombre = input(
                f"\n¿Cuál es la actividad {numero}? "
                "(describe brevemente qué haces): "
            ).strip()
            if not nombre:
                print("⚠️ Por favor, ingresa una descripción.")

        frecuencia = None
        opciones_str = "  ".join(
            f"{i+1}) {f}" for i, f in enumerate(self.FRECUENCIAS_VALIDAS)
        )
        print(f"\n¿Con qué frecuencia realizas esta actividad?")
        print(f"  {opciones_str}")
        while frecuencia is None:
            resp = input("Selecciona el número o escribe la frecuencia: ").strip()
            if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                frecuencia = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
            elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                frecuencia = resp.capitalize()
            else:
                print(
                    "⚠️ Opción no válida. "
                    f"Elige un número del 1 al {len(self.FRECUENCIAS_VALIDAS)}."
                )

        volumen = None
        print(
            f"\n¿Cuántas veces realizas esta actividad por periodo "
            f"({frecuencia.lower()})?"
        )
        print("  (Ej: si es Diario y la haces 3 veces al día → escribe 3)")
        while volumen is None:
            resp = input("Volumen: ").strip()
            if resp.isdigit() and int(resp) > 0:
                volumen = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        duracion_min = None
        print(
            "\n¿Cuántos minutos te toma completar UNA ejecución "
            "de esta actividad?"
        )
        while duracion_min is None:
            resp = input("Duración (minutos): ").strip()
            if resp.isdigit() and int(resp) > 0:
                duracion_min = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        autonomia_opciones = list(range(5, 105, 5))
        autonomia = None
        print("\n¿Qué porcentaje de esta actividad recae sobre ti?")
        fila = "  " + "  ".join(f"{p}%" for p in autonomia_opciones)
        print(f"  Opciones disponibles (de 5 en 5):\n{fila}")
        print(
            "  (Ej: si eres el único responsable → 100%  |  "
            "si compartes al 50% con otro → 50%)"
        )
        while autonomia is None:
            resp = input("Autonomía (%): ").strip().replace("%", "").strip()
            if resp.isdigit() and int(resp) in autonomia_opciones:
                autonomia = int(resp)
            else:
                print(
                    "⚠️ Elige un valor de 5 en 5 entre 5% y 100%. "
                    "Ej: 5, 10, 15, ... , 95, 100."
                )

        proceso_area = self._preguntar_proceso_area(procesos_registrados)

        observaciones = input(
            "\n¿Tienes alguna observación adicional sobre esta actividad? "
            "(presiona Enter si no): "
        ).strip()

        actividad_borrador = {
            "nombre":        nombre,
            "frecuencia":    frecuencia,
            "volumen":       volumen,
            "duracion_min":  duracion_min,
            "autonomia":     autonomia,
            "proceso_area":  proceso_area,
            "observaciones": observaciones,
        }

        print("\n🔍 Verificando proceso del área y observaciones...")
        actividad_borrador = self._verificar_proceso_y_observaciones(
            actividad_borrador, contexto
        )

        return actividad_borrador

    # -----------------------------------------------------------------------
    # E.0  HELPERS RÁPIDOS (sin LLM)
    # -----------------------------------------------------------------------
    def _calcular_minutos_diarios_acumulados(self, actividades):
        FACTOR_FRECUENCIA = {
            "Diario": 1, "Semanal": 1/5, "Quincenal": 1/10,
            "Mensual": 1/21, "Bimensual": 1/42, "Trimestral": 1/63,
            "Semestral": 1/126, "Anual": 1/252,
        }
        total = 0.0
        for act in actividades:
            factor = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            total += act["duracion_min"] * act["volumen"] * factor
        return round(total, 2)

    def _es_duplicada(self, nombre_nuevo, actividades_existentes, umbral=0.75):
        from difflib import SequenceMatcher
        nombre_norm = nombre_nuevo.strip().lower()
        for act in actividades_existentes:
            existente_norm = act["nombre"].strip().lower()
            ratio = SequenceMatcher(None, nombre_norm, existente_norm).ratio()
            if ratio >= umbral:
                return True, act["nombre"]
        return False, None

    # -----------------------------------------------------------------------
    # E. FLUJO PRINCIPAL CONVERSACIONAL
    # -----------------------------------------------------------------------
    def _capturar_actividades_conversacional_con_procesos(
        self, contexto, procesos_iniciales=None
    ):
        return self.capturar_actividades_conversacional(
            contexto, procesos_iniciales=procesos_iniciales or []
        )

    def capturar_actividades_conversacional(self, contexto, procesos_iniciales=None):
        if procesos_iniciales is None:
            procesos_iniciales = []

        JORNADA_MIN = 8.5 * 60

        print("\n" + "═" * 60)
        print("  🗂️  REGISTRO DE ACTIVIDADES — MODO CONVERSACIONAL")
        print("═" * 60)
        print(
            "\nA continuación te haré preguntas sobre cada actividad que realizas.\n"
            "Puedes registrar todas las que quieras.\n"
            "Cuando termines, escribe 'no' cuando te pregunte si hay otra actividad."
        )

        actividades = []
        numero = 1
        tope_alcanzado = False

        while True:
            while True:
                procesos_registrados = list(procesos_iniciales) + [
                    act.get("proceso_area", "") for act in actividades
                ]

                actividad = self._preguntar_actividad(
                    numero,
                    procesos_registrados=procesos_registrados,
                    contexto=contexto,
                )

                es_dup, nombre_similar = self._es_duplicada(
                    actividad["nombre"], actividades
                )
                if es_dup:
                    print("\n" + "─" * 60)
                    print("⚠️  POSIBLE ACTIVIDAD DUPLICADA")
                    print("─" * 60)
                    print(
                        f"   La actividad \"{actividad['nombre']}\" es muy similar\n"
                        f"   a \"{nombre_similar}\", que ya fue registrada."
                    )
                    decision = input(
                        "\n   ¿Es realmente una actividad diferente? (si / no): "
                    ).strip().lower()
                    if decision != "si":
                        print("🔄 Descartada. Ingresa una actividad distinta.\n")
                        continue

                print("\n🔍 Evaluando relevancia y completitud de la actividad...")
                evaluacion = self._evaluar_relevancia_y_detalle(
                    actividad,
                    contexto.get("cargo", ""),
                    contexto.get("vicepresidencia", "")
                )
                accion, actividad = self._manejar_alerta_relevancia(
                    actividad, evaluacion
                )
                if accion == "descartar":
                    print("🔄 Ingresa una nueva actividad.\n")
                    continue

                min_acum_previo = self._calcular_minutos_diarios_acumulados(actividades)
                min_disponibles = max(JORNADA_MIN - min_acum_previo, 0)

                print("\n🔍 Revisando coherencia de los datos ingresados...")
                hay_incoherencia, explicacion, recomendaciones = (
                    self._validar_coherencia_actividad_ia(
                        actividad,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                )

                if hay_incoherencia:
                    actividad = self._mostrar_incoherencia_y_corregir(
                        actividad,
                        explicacion,
                        recomendaciones,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                else:
                    print("✅ Coherencia OK — los datos ingresados son consistentes.")

                confirmada = self._confirmar_actividad(
                    actividad           = actividad,
                    numero              = numero,
                    actividades_previas = actividades,
                    contexto            = contexto,
                )
                if confirmada:
                    actividades.append(actividad)
                    print(f"\n✅ Actividad {numero} registrada.")
                    break
                else:
                    print(
                        f"\n🔄 Volvemos a registrar la actividad {numero}. "
                        "Ingresa los datos de nuevo.\n"
                    )

            min_acum   = self._calcular_minutos_diarios_acumulados(actividades)
            horas_acum = min_acum / 60
            min_rest   = max(JORNADA_MIN - min_acum, 0)
            horas_rest = min_rest / 60

            pct     = min(min_acum / JORNADA_MIN, 1.0)
            bloques = int(pct * 20)
            barra   = "█" * bloques + "░" * (20 - bloques)
            print(
                f"\n  ⏱️  Carga acumulada: [{barra}] "
                f"{horas_acum:.1f} h de 8.5 h hábiles  "
                f"({horas_rest:.1f} h disponibles)"
            )

            if min_acum >= JORNADA_MIN and not tope_alcanzado:
                tope_alcanzado = True
                print("\n" + "⚠️ " * 20)
                print(
                    "  ATENCIÓN: Las actividades registradas ya ocupan\n"
                    "  TODA la jornada laboral disponible (8.5 horas).\n"
                    "  Si agregas más, la carga superará la capacidad diaria."
                )
                print("⚠️ " * 20)
                continuar = input(
                    "\n  ¿Deseas seguir registrando actividades de todas formas? "
                    "(si / no): "
                ).strip().lower()
                if continuar != "si":
                    break

            elif min_acum >= JORNADA_MIN and tope_alcanzado:
                exceso_min = round(min_acum - JORNADA_MIN, 1)
                print(
                    f"  ⚠️  Ya superaste la jornada diaria en "
                    f"~{exceso_min} min equivalentes."
                )

            print("\n" + "─" * 60)
            while True:
                otra = input(
                    "¿Tienes otra actividad para registrar? (si / no): "
                ).strip().lower()
                if otra in ("si", "no"):
                    break
                print("⚠️ Opción no válida. Por favor escribe 'si' o 'no'.")

            if otra != "si":
                break

            numero += 1

        print("\n" + "═" * 60)
        print(f"  ✅  Se registraron {len(actividades)} actividad(es) en total.")
        print("═" * 60)
        return actividades

    # -----------------------------------------------------------------------
    # E.1  CAPTURAR CONTEXTO DEL COLABORADOR — búsqueda en Excel por documento
    # -----------------------------------------------------------------------
    def capturar_contexto_conversacional(
        self,
        ruta_excel_planta: str = "Corte_Planta_30Ene2026_Organigrama.xlsx",
    ):
        """
        Obtiene cargo, vicepresidencia y gerencia del colaborador buscando
        su número de documento en un archivo Excel de planta.

        CAMBIO: si el número de documento no se encuentra en el Excel,
        ya NO se ofrece la opción de ingresar los datos manualmente.
        Solo se permite reintentar con otro número de documento.
        El ingreso manual (_contexto_manual_fallback) sigue disponible
        únicamente cuando el archivo de planta no existe o no puede cargarse.
        """
        _MAPA_COLUMNAS = {
            "documento":       "cedula",
            "cedula":          "cedula",
            "cédula":          "cedula",
            "cc":              "cedula",
            "id":              "cedula",
            "identificacion":  "cedula",
            "identificación":  "cedula",
            "num_doc":         "cedula",
            "numero_cedula":   "cedula",
            "número_cédula":   "cedula",
            "num_cedula":      "cedula",
            "nombre":          "nombre_completo",
            "nombre_completo": "nombre_completo",
            "nombres":         "nombre_completo",
            "colaborador":     "nombre_completo",
            "empleado":        "nombre_completo",
            "funcionario":     "nombre_completo",
            "cargo":           "cargo",
            "puesto":          "cargo",
            "posicion":        "cargo",
            "posición":        "cargo",
            "titulo":          "cargo",
            "título":          "cargo",
            "rol":             "cargo",
            "vicepresidencia": "vicepresidencia",
            "vp":              "vicepresidencia",
            "gerencia_padre":  "vicepresidencia",
            "area_padre":      "vicepresidencia",
            "direccion":       "vicepresidencia",
            "dirección":       "vicepresidencia",
            "division":        "vicepresidencia",
            "división":        "vicepresidencia",
            "gerencia":              "gerencia",
            "area":                  "gerencia",
            "área":                  "gerencia",
            "subarea":               "gerencia",
            "subárea":               "gerencia",
            "departamento":          "gerencia",
            "dpto":                  "gerencia",
            "unidad":                "gerencia",
            "unidad_organizacional": "gerencia",
            "gerencia_area":         "gerencia",
            "nombre_gerencia":       "gerencia",
            "nom_gerencia":          "gerencia",
            "nombre_area":           "gerencia",
            "nom_area":              "gerencia",
            "direccion_gerencia":    "gerencia",
            "email":                 "email",
            "correo":          "email",
            "correo_electronico": "email",
        }

        def _normalizar(texto):
            import unicodedata
            texto = str(texto).strip().lower()
            texto = unicodedata.normalize("NFKD", texto)
            texto = "".join(c for c in texto if not unicodedata.combining(c))
            texto = re.sub(r"[\s\-\/]+", "_", texto)
            texto = re.sub(r"[^a-z0-9_]", "", texto)
            return texto

        def _cargar_planta(ruta):
            ext = os.path.splitext(ruta)[1].lower()
            filas_raw = []

            if ext in (".xlsx", ".xls"):
                wb = load_workbook(ruta, data_only=True)
                ws = wb.active
                iter_filas = ws.iter_rows(values_only=True)
                encabezados_raw = next(iter_filas, None)
                if not encabezados_raw:
                    return []
                encabezados = [
                    _MAPA_COLUMNAS.get(_normalizar(str(h or "")),
                                       _normalizar(str(h or "")))
                    for h in encabezados_raw
                ]
                # Diagnóstico: mostrar columnas no reconocidas por el mapa
                no_mapeadas = [
                    str(h) for h in encabezados_raw
                    if h and _normalizar(str(h)) not in _MAPA_COLUMNAS
                ]
                campos_canonicos = set(encabezados)
                if "gerencia" not in campos_canonicos and no_mapeadas:
                    print(
                        f"\n  ⚠️  No se detectó la columna 'gerencia' en el Excel.\n"
                        f"     Columnas no reconocidas: {no_mapeadas}\n"
                        f"     Si alguna corresponde a gerencia, agrégala al _MAPA_COLUMNAS."
                    )
                for fila in iter_filas:
                    if all(v is None or str(v).strip() == "" for v in fila):
                        continue
                    filas_raw.append(
                        {enc: (str(val).strip() if val is not None else "")
                         for enc, val in zip(encabezados, fila)}
                    )

            elif ext == ".csv":
                import csv
                import unicodedata
                for enc in ("utf-8-sig", "utf-8", "latin-1"):
                    try:
                        with open(ruta, encoding=enc) as f:
                            sample = f.read(2048)
                        break
                    except UnicodeDecodeError:
                        continue
                dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
                with open(ruta, encoding=enc, newline="") as f:
                    reader = csv.DictReader(f, dialect=dialect)
                    mapa = {
                        h: _MAPA_COLUMNAS.get(_normalizar(h), _normalizar(h))
                        for h in (reader.fieldnames or [])
                    }
                    for fila in reader:
                        if all(v.strip() == "" for v in fila.values()):
                            continue
                        filas_raw.append(
                            {mapa[k]: v.strip() for k, v in fila.items()}
                        )
            else:
                raise ValueError(f"Formato no soportado: '{ext}'. Usa .xlsx o .csv")

            return filas_raw

        def _buscar_en_planta(planta, numero_doc):
            doc_norm = re.sub(r"[\s\-]", "", numero_doc).lstrip("0") or "0"
            for fila in planta:
                doc_fila = re.sub(r"[\s\-]", "", fila.get("cedula", "")).lstrip("0") or "0"
                if doc_fila == doc_norm:
                    return fila
            return None

        # ── Inicio del flujo conversacional ───────────────────────────────
        print("\n" + "═" * 60)
        print("  👤  IDENTIFICACIÓN DEL COLABORADOR")
        print("═" * 60)

        # Intentar cargar el Excel de planta
        planta = None
        if os.path.exists(ruta_excel_planta):
            try:
                planta = _cargar_planta(ruta_excel_planta)
                print(f"\n  ✅ Planta cargada: {len(planta)} colaborador(es) registrado(s).")
            except Exception as e:
                print(f"\n  ⚠️  No se pudo leer el archivo de planta: {e}")
                planta = None
        else:
            print(
                f"\n  ⚠️  No se encontró el archivo de planta "
                f"'{ruta_excel_planta}'.\n"
                "     Asegúrate de que el archivo exista en el directorio actual\n"
                "     o pasa la ruta correcta a capturar_contexto_conversacional()."
            )

        # Si la planta no está disponible → fallback manual (único caso permitido)
        if not planta:
            print("     Se usará ingreso manual como alternativa.")
            return self._contexto_manual_fallback()

        # ── Búsqueda por número de documento ──────────────────────────────
        # CAMBIO: si no se encuentra el documento, solo se ofrece reintentar.
        # Ya no se ofrece la opción de ingresar datos manualmente.
        while True:
            numero_doc = ""
            while not numero_doc:
                numero_doc = input(
                    "\n  Ingresa tu número de documento (sin puntos ni espacios): "
                ).strip()
                if not numero_doc:
                    print("  ⚠️ Debes ingresar tu número de documento.")

            colaborador = _buscar_en_planta(planta, numero_doc)

            if colaborador:
                # Mostrar datos encontrados
                print("\n" + "─" * 60)
                print("  ✅  Colaborador encontrado:")
                print("─" * 60)
                if colaborador.get("nombre_completo"):
                    print(f"  Nombre          : {colaborador['nombre_completo']}")
                print(f"  Cargo           : {colaborador.get('cargo', '—')}")
                print(f"  Vicepresidencia : {colaborador.get('vicepresidencia', '—')}")
                if colaborador.get("gerencia"):
                    print(f"  Gerencia        : {colaborador['gerencia']}")
                if colaborador.get("email"):
                    print(f"  Email           : {colaborador['email']}")
                print("─" * 60)

                while True:
                    confirma = input(
                        "\n  ¿Es tu información correcta? (si / no): "
                    ).strip().lower()
                    if confirma in ("si", "no"):
                        break
                    print("  ⚠️ Por favor escribe 'si' o 'no'.")

                if confirma == "si":
                    nombre_display = colaborador.get("nombre_completo") or numero_doc
                    print(
                        f"\n  ✅ Sesión iniciada: {nombre_display} — "
                        f"{colaborador.get('cargo', '')}"
                    )
                    return {
                        "cedula":          numero_doc,
                        "nombre_completo": colaborador.get("nombre_completo", ""),
                        "cargo":           colaborador.get("cargo", ""),
                        "vicepresidencia": colaborador.get("vicepresidencia", ""),
                        "gerencia":        colaborador.get("gerencia", ""),
                        "email":           colaborador.get("email", ""),
                    }

                # Si el colaborador dice "no" (datos desactualizados en planta)
                print(
                    "\n  ℹ️  Si los datos no son correctos, solicita al encargado\n"
                    "     del proceso que actualice el archivo de planta.\n"
                    "     Puedes intentar con otro número de documento."
                )
                # Vuelve al inicio del bucle para pedir otro número

            else:
                # Documento no encontrado — SOLO se ofrece reintentar
                print(
                    f"\n  ⚠️  No se encontró ningún colaborador con\n"
                    f"     documento '{numero_doc}' en el archivo de planta.\n"
                    f"\n  ℹ️  Verifica que el número sea correcto (sin puntos,\n"
                    f"     comas ni espacios) y vuelve a intentarlo.\n"
                    f"     Si el problema persiste, comunícate con el encargado\n"
                    f"     del proceso para que actualice el archivo de planta."
                )
                # El bucle while True hace que se vuelva a pedir el documento

    def _contexto_manual_fallback(self) -> dict:
        """
        Ingreso manual de datos como fallback excepcional.
        Se usa ÚNICAMENTE cuando el archivo Excel de planta no existe
        o no puede cargarse. Ya NO se invoca cuando el documento
        no se encuentra en la planta.
        """
        print("\n" + "─" * 60)
        print("  ⚠️  INGRESO MANUAL — DATOS NO VERIFICADOS")
        print(
            "  Estos datos no están respaldados por la planta oficial.\n"
            "  El encargado del proceso debe actualizar el archivo Excel."
        )
        print("─" * 60)

        numero_doc = input("\n  Número de documento: ").strip()

        nombre = ""
        while not nombre:
            nombre = input("  Nombre completo    : ").strip()
            if not nombre:
                print("  ⚠️ Ingresa tu nombre.")

        cargo = ""
        while not cargo:
            cargo = input("  Cargo actual       : ").strip()
            if not cargo:
                print("  ⚠️ Ingresa tu cargo.")

        vicepresidencia = ""
        while not vicepresidencia:
            vicepresidencia = input("  Vicepresidencia    : ").strip()
            if not vicepresidencia:
                print("  ⚠️ Ingresa la vicepresidencia.")

        gerencia = input("  Gerencia (opcional): ").strip()

        print("\n" + "─" * 60)
        print("  Datos ingresados manualmente:")
        print(f"  Documento       : {numero_doc or '(no ingresado)'}")
        print(f"  Nombre          : {nombre}")
        print(f"  Cargo           : {cargo}")
        print(f"  Vicepresidencia : {vicepresidencia}")
        if gerencia:
            print(f"  Gerencia        : {gerencia}")
        print("─" * 60)

        while True:
            confirma = input("\n  ¿Confirmas estos datos? (si / no): ").strip().lower()
            if confirma == "si":
                break
            if confirma == "no":
                return self._contexto_manual_fallback()
            print("  ⚠️ Por favor escribe 'si' o 'no'.")

        print("  ✅ Datos registrados (modo manual).")
        return {
            "cedula":          numero_doc,
            "nombre_completo": nombre,
            "cargo":           cargo,
            "vicepresidencia": vicepresidencia,
            "gerencia":        gerencia,
            "email":           "",
            "_ingreso_manual": True,
        }

    # -----------------------------------------------------------------------
    # ★ SELECCIÓN DE MODO AL INICIO DEL PIPELINE
    # -----------------------------------------------------------------------
    def _seleccionar_modo_inicio(self):
        print("\n" + "═" * 60)
        print("  🚀  BIENVENIDO — LEVANTAMIENTO DE CARGAS LABORALES")
        print("═" * 60)
        print(
            "\n¿Cómo deseas iniciar el proceso?\n"
            "\n  1) Solo registrar mis actividades manualmente"
            "\n  2) Registrar actividades y además cargar la transcripción"
            "\n     de una reunión de Microsoft Teams (archivo .vtt)"
        )

        while True:
            opcion = input("\nSelecciona una opción (1 / 2): ").strip()
            if opcion == "1":
                print("\n✅ Modo seleccionado: Registro manual de actividades.\n")
                return "manual"
            elif opcion == "2":
                print(
                    "\n✅ Modo seleccionado: Registro manual + "
                    "transcripción de Teams.\n"
                )
                return "con_transcripcion"
            else:
                print("⚠️ Opción no válida. Por favor selecciona 1 o 2.")

    # -----------------------------------------------------------------------
    # ★ CONSOLIDACIÓN DE ACTIVIDADES REPETIDAS EN LA TRANSCRIPCIÓN
    # -----------------------------------------------------------------------
    def _consolidar_duplicados_vtt(self, actividades_crudas):
        import math

        if len(actividades_crudas) <= 1:
            return actividades_crudas

        print("\n🔍 Analizando actividades repetidas en la transcripción...")

        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.

Se te entrega una lista de actividades extraídas de una grabación de
Microsoft Teams. Algunas pueden estar repetidas (la misma tarea realizada
varias veces durante la reunión con texto ligeramente distinto).

Tu tarea:
1. Identifica grupos de actividades que sean la MISMA tarea.
   Considera como "misma tarea" cuando el texto es semánticamente
   equivalente aunque las palabras difieran levemente
   (ej: "revisión de informe" y "revisar informe" → mismo grupo).
2. Asigna a cada actividad un "grupo_id" (entero, empezando en 0).
   Actividades únicas (sin repetición) van cada una en su propio grupo.
3. Para cada grupo propón un "nombre_consolidado" limpio y descriptivo.

⚠️ REGLAS ESTRICTAS:
- Devuelve EXCLUSIVAMENTE un JSON válido, sin texto ni markdown.
- La lista de salida debe tener EXACTAMENTE el mismo número de elementos
  que la lista de entrada, en el mismo orden.

Formato de salida:
[
  {{
    "indice_original": 0,
    "grupo_id": 0,
    "nombre_consolidado": "nombre limpio del grupo"
  }},
  ...
]

Actividades de entrada (índice, texto, duración en minutos):
{json.dumps(
    [{"indice": i, "texto": a.get("actividad",""), "duracion_min": a.get("duracion_min", 0)}
     for i, a in enumerate(actividades_crudas)],
    indent=2, ensure_ascii=False
)}
"""
        try:
            reply = llm.invoke(prompt)
            clasificaciones = self._parse_json_seguro(reply.content)
        except Exception:
            print("⚠️ No se pudo analizar duplicados. Se conservan todas las actividades.")
            return actividades_crudas

        grupos = {}
        for clf in clasificaciones:
            idx    = clf.get("indice_original", 0)
            gid    = clf.get("grupo_id", idx)
            nombre = clf.get("nombre_consolidado", "")
            if gid not in grupos:
                grupos[gid] = {"nombre": nombre, "items": []}
            if idx < len(actividades_crudas):
                grupos[gid]["items"].append(actividades_crudas[idx])

        grupos_con_dup = {gid: g for gid, g in grupos.items() if len(g["items"]) > 1}
        grupos_unicos  = {gid: g for gid, g in grupos.items() if len(g["items"]) == 1}

        if not grupos_con_dup:
            print("✅ No se detectaron actividades repetidas.")
            return actividades_crudas

        def duracion_representativa(duraciones):
            n = len(duraciones)
            if n <= 2:
                duraciones_sorted = sorted(duraciones)
                mid = n // 2
                if n % 2 == 0:
                    return round((duraciones_sorted[mid - 1] + duraciones_sorted[mid]) / 2)
                return duraciones_sorted[mid]
            corte = max(1, math.floor(n * 0.20))
            recortadas = sorted(duraciones)[corte: n - corte]
            mid = len(recortadas) // 2
            if len(recortadas) % 2 == 0:
                return round((recortadas[mid - 1] + recortadas[mid]) / 2)
            return recortadas[mid]

        print("\n" + "═" * 60)
        print("  🔁  ACTIVIDADES REPETIDAS DETECTADAS")
        print("═" * 60)
        print(
            f"\n  Se encontraron {len(grupos_con_dup)} grupo(s) de actividades "
            "que parecen repetirse.\n"
            "  Por cada grupo se propondrá consolidarlas en UNA sola actividad\n"
            "  usando la duración más representativa del grupo.\n"
        )

        actividades_resultado = []

        for gid, grupo in sorted(grupos_con_dup.items()):
            items   = grupo["items"]
            nombre  = grupo["nombre"]
            durs    = [a.get("duracion_min", 0) for a in items]
            dur_rep = duracion_representativa(durs)

            print("─" * 60)
            print(f"  Grupo: \"{nombre}\"")
            print(f"  Apariciones: {len(items)}")
            print(f"  Duraciones individuales (min): {durs}")
            print(
                f"  Duración representativa propuesta: {dur_rep} min\n"
                f"  (mediana recortada — descarta tiempos atípicos)"
            )
            print("─" * 60)

            while True:
                resp = input(
                    "  ¿Consolidar en UNA actividad con esa duración? (si / no): "
                ).strip().lower()
                if resp in ("si", "no"):
                    break
                print("  ⚠️ Por favor escribe 'si' o 'no'.")

            if resp == "si":
                actividades_resultado.append({
                    "actividad":    nombre,
                    "inicio":       items[0].get("inicio", ""),
                    "fin":          items[-1].get("fin", ""),
                    "duracion_min": dur_rep,
                    "_consolidada": True,
                    "_ocurrencias": len(items),
                    "_duraciones_originales": durs,
                })
                print(
                    f"  ✅ Consolidada como \"{nombre}\" "
                    f"({len(items)} ocurrencias → {dur_rep} min)."
                )
            else:
                for act in items:
                    actividades_resultado.append(act)
                print(f"  ↩️ Se conservan las {len(items)} ocurrencias por separado.")

        for gid, grupo in sorted(grupos_unicos.items()):
            actividades_resultado.extend(grupo["items"])

        def _ts_to_secs(ts):
            try:
                partes = ts.split(":")
                return int(partes[0]) * 3600 + int(partes[1]) * 60 + int(partes[2])
            except Exception:
                return 0

        actividades_resultado.sort(key=lambda a: _ts_to_secs(a.get("inicio", "0:0:0")))

        print("\n" + "═" * 60)
        n_orig  = len(actividades_crudas)
        n_final = len(actividades_resultado)
        print(
            f"  ✅  Resultado: {n_orig} actividades detectadas → "
            f"{n_final} actividades tras consolidación."
        )
        print("═" * 60)

        return actividades_resultado

    def _cargar_vtt_conversacional(self, contexto, procesos_registrados=None):
        if procesos_registrados is None:
            procesos_registrados = []

        print("\n" + "─" * 60)
        print("  📂  CARGA DE TRANSCRIPCIÓN DE TEAMS (.vtt)")
        print("─" * 60)

        while True:
            ruta = input(
                "\nIngresa la ruta del archivo .vtt "
                "(o presiona Enter para omitir este paso): "
            ).strip()

            if not ruta:
                print("⚠️ No se cargó ningún archivo. Continuando sin transcripción.")
                return []

            try:
                with open(ruta, "r", encoding="utf-8") as f:
                    contenido = f.read()
                actividades_crudas = self.procesar_archivo(contenido)
                break
            except FileNotFoundError:
                print(f"⚠️ Archivo no encontrado: {ruta}. Intenta de nuevo.")
            except Exception as e:
                print(f"⚠️ Error al procesar el archivo: {e}. Intenta de nuevo.")

        if not actividades_crudas:
            print("\n⚠️ No se detectaron actividades en la transcripción.")
            return []

        actividades_crudas = self._consolidar_duplicados_vtt(actividades_crudas)

        print("\n" + "═" * 60)
        print("  📋  RESUMEN DE ACTIVIDADES DETECTADAS EN LA TRANSCRIPCIÓN")
        print("═" * 60)
        print(f"  Se encontraron {len(actividades_crudas)} actividad(es):\n")

        col_n    = len(str(len(actividades_crudas)))
        col_dur  = 10
        col_ocu  = 12
        ancho_desc = 40

        encabezado = (
            f"  {'#':>{col_n}}  "
            f"{'Actividad detectada':<{ancho_desc}}  "
            f"{'Duración':>{col_dur}}  "
            f"{'Ocurrencias':>{col_ocu}}"
        )
        print(encabezado)
        print("  " + "─" * (col_n + 2 + ancho_desc + 2 + col_dur + 2 + col_ocu))

        for i, act in enumerate(actividades_crudas, start=1):
            desc = act.get("actividad", "Sin descripción")
            if len(desc) > ancho_desc:
                desc = desc[: ancho_desc - 3] + "..."
            dur     = act.get("duracion_min", 0)
            dur_txt = f"{dur} min"
            ocu     = act.get("_ocurrencias", 1)
            ocu_txt = f"{'★ ' + str(ocu) + 'x' if ocu > 1 else '—':>{col_ocu}}"
            print(
                f"  {i:>{col_n}}  "
                f"{desc:<{ancho_desc}}  "
                f"{dur_txt:>{col_dur}}  "
                f"{ocu_txt}"
            )

        print("  " + "─" * (col_n + 2 + ancho_desc + 2 + col_ocu))
        print(
            "\n  Nota: 'Actividad detectada' corresponde al fragmento de texto\n"
            "  de la grabación en el que se identificó el inicio de una tarea.\n"
            "  La duración es el tiempo representativo por ejecución.\n"
            "  ★ = actividad consolidada (aparecía varias veces en la grabación)."
        )
        print("═" * 60)

        while True:
            completar = input(
                "\n¿Deseas completar los campos faltantes de estas actividades "
                "para incluirlas en el levantamiento? (si / no): "
            ).strip().lower()
            if completar in ("si", "no"):
                break
            print("⚠️ Por favor escribe 'si' o 'no'.")

        if completar != "si":
            print(
                "\nℹ️  Las actividades de la transcripción se usarán solo como\n"
                "   referencia y NO se incluirán en el Excel."
            )
            return []

        print("\n" + "═" * 60)
        print("  ✍️  COMPLETAR CAMPOS DE ACTIVIDADES DE LA TRANSCRIPCIÓN")
        print("═" * 60)
        print(
            "\n  Para cada actividad detectada podrás:\n"
            "  • Ajustar su nombre si el texto extraído no es claro.\n"
            "  • Ingresar los campos operativos que faltan.\n"
            "  • Descartarla si no deseas incluirla."
        )

        actividades_completadas = []

        for i, act_cruda in enumerate(actividades_crudas, start=1):
            print("\n" + "─" * 60)
            print(f"  ACTIVIDAD {i} DE {len(actividades_crudas)}")
            print("─" * 60)
            print(f"  Texto extraído : {act_cruda.get('actividad', '')}")
            print(f"  Duración       : {act_cruda.get('duracion_min', 0)} min")
            if not act_cruda.get("_consolidada"):
                print(f"  Inicio         : {act_cruda.get('inicio', '')}  "
                      f"Fin: {act_cruda.get('fin', '')}")
            else:
                print(
                    f"  ★ Consolidada de {act_cruda.get('_ocurrencias', '?')} "
                    f"ocurrencias — duraciones originales: "
                    f"{act_cruda.get('_duraciones_originales', [])} min"
                )

            while True:
                incluir = input(
                    "\n  ¿Incluir esta actividad? (si / no): "
                ).strip().lower()
                if incluir in ("si", "no"):
                    break
                print("  ⚠️ Por favor escribe 'si' o 'no'.")

            if incluir != "si":
                print("  🗑️  Actividad descartada.")
                continue

            while True:
                nombre_sugerido = act_cruda.get("actividad", "").strip()
                print(f"\n  Nombre sugerido: \"{nombre_sugerido}\"")
                ajuste = input(
                    "  ¿Quieres ajustar el nombre? (escribe el nuevo nombre "
                    "o presiona Enter para conservarlo): "
                ).strip()
                nombre = ajuste if ajuste else nombre_sugerido

                dur_vtt = act_cruda.get("duracion_min", 0)
                print(f"\n  La duración detectada es {dur_vtt} min.")
                while True:
                    resp = input(
                        "  ¿Es correcta? (si) o ingresa la duración real en minutos: "
                    ).strip()
                    if resp.lower() == "si" or resp == "":
                        duracion_min = dur_vtt
                        break
                    elif resp.isdigit() and int(resp) > 0:
                        duracion_min = int(resp)
                        break
                    else:
                        print("  ⚠️ Ingresa 'si' o un número entero mayor a 0.")

                frecuencia = None
                opciones_str = "  ".join(
                    f"{j+1}) {f}" for j, f in enumerate(self.FRECUENCIAS_VALIDAS)
                )
                print(f"\n  ¿Con qué frecuencia realizas esta actividad?")
                print(f"    {opciones_str}")
                while frecuencia is None:
                    resp = input("  Selecciona el número o escribe la frecuencia: ").strip()
                    if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                        frecuencia = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
                    elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                        frecuencia = resp.capitalize()
                    else:
                        print(
                            f"  ⚠️ Opción no válida. Elige un número del "
                            f"1 al {len(self.FRECUENCIAS_VALIDAS)}."
                        )

                volumen = None
                print(
                    f"\n  ¿Cuántas veces realizas esta actividad por periodo "
                    f"({frecuencia.lower()})?"
                )
                while volumen is None:
                    resp = input("  Volumen: ").strip()
                    if resp.isdigit() and int(resp) > 0:
                        volumen = int(resp)
                    else:
                        print("  ⚠️ Ingresa un número entero mayor a 0.")

                autonomia_opciones = list(range(5, 105, 5))
                autonomia = None
                fila_aut = "    " + "  ".join(f"{p}%" for p in autonomia_opciones)
                print("\n  ¿Qué porcentaje de esta actividad recae sobre ti?")
                print(f"  Opciones (de 5 en 5):\n{fila_aut}")
                while autonomia is None:
                    resp = input("  Autonomía (%): ").strip().replace("%", "").strip()
                    if resp.isdigit() and int(resp) in autonomia_opciones:
                        autonomia = int(resp)
                    else:
                        print("  ⚠️ Elige un valor de 5 en 5 entre 5% y 100%.")

                procesos_acumulados = list(procesos_registrados) + [
                    a.get("proceso_area", "")
                    for a in actividades_completadas
                    if a.get("proceso_area")
                ]
                proceso_area = self._preguntar_proceso_area(procesos_acumulados)

                observaciones = input(
                    "\n  Observaciones adicionales (Enter para omitir): "
                ).strip()

                actividad_borrador_vtt = {
                    "nombre":        nombre,
                    "frecuencia":    frecuencia,
                    "volumen":       volumen,
                    "duracion_min":  duracion_min,
                    "autonomia":     autonomia,
                    "proceso_area":  proceso_area,
                    "observaciones": observaciones,
                }
                print("\n🔍 Verificando proceso del área y observaciones...")
                actividad_borrador_vtt = self._verificar_proceso_y_observaciones(
                    actividad_borrador_vtt, contexto
                )
                nombre       = actividad_borrador_vtt["nombre"]
                proceso_area = actividad_borrador_vtt["proceso_area"]
                observaciones = actividad_borrador_vtt["observaciones"]

                actividad = {
                    "nombre":        nombre,
                    "frecuencia":    frecuencia,
                    "volumen":       volumen,
                    "duracion_min":  duracion_min,
                    "autonomia":     autonomia,
                    "proceso_area":  proceso_area,
                    "observaciones": observaciones,
                    "_origen":       "transcripcion_teams",
                    **{k: act_cruda[k] for k in (
                        "_consolidada", "_ocurrencias", "_duraciones_originales"
                    ) if k in act_cruda},
                }

                print("\n🔍 Evaluando relevancia y completitud de la actividad...")
                evaluacion = self._evaluar_relevancia_y_detalle(
                    actividad,
                    contexto.get("cargo", ""),
                    contexto.get("vicepresidencia", "")
                )
                accion, actividad = self._manejar_alerta_relevancia(
                    actividad, evaluacion
                )
                if accion == "descartar":
                    print("  🗑️  Actividad descartada por el análisis de relevancia.")
                    actividad = None
                    break

                min_acum_previo = self._calcular_minutos_diarios_acumulados(
                    actividades_completadas
                )
                min_disponibles = max(8.5 * 60 - min_acum_previo, 0)

                print("\n🔍 Revisando coherencia de los datos ingresados...")
                hay_incoherencia, explicacion, recomendaciones = (
                    self._validar_coherencia_actividad_ia(
                        actividad,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                )
                if hay_incoherencia:
                    actividad = self._mostrar_incoherencia_y_corregir(
                        actividad,
                        explicacion,
                        recomendaciones,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                else:
                    print("✅ Coherencia OK — los datos ingresados son consistentes.")

                confirmada = self._confirmar_actividad(
                    actividad           = actividad,
                    numero              = i,
                    actividades_previas = actividades_completadas,
                    contexto            = contexto,
                )
                if confirmada:
                    break

            if actividad is not None:
                actividades_completadas.append(actividad)
                print(f"  ✅ Actividad {i} completada y añadida al levantamiento.")

        print("\n" + "═" * 60)
        print(
            f"  ✅  {len(actividades_completadas)} de {len(actividades_crudas)} "
            "actividad(es) de la transcripción serán incluidas."
        )
        print("═" * 60)

        return actividades_completadas

    # -----------------------------------------------------------------------
    # G. GENERAR EXCEL DESCARGABLE CON LOS DATOS DEL COLABORADOR
    # -----------------------------------------------------------------------
    def generar_excel_actividades(
        self,
        actividades,
        contexto,
        ruta_plantilla=None,
        ruta_salida="levantamiento_cargas.xlsx"
    ):
        from openpyxl import Workbook, load_workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        vicepresidencia = contexto.get("vicepresidencia", "")

        if ruta_plantilla:
            wb = load_workbook(ruta_plantilla)
            ws = wb["FORMATO"] if "FORMATO" in wb.sheetnames else wb.active

            fila_encabezado = None
            for fila in ws.iter_rows():
                for celda in fila:
                    if (
                        celda.value
                        and "DESCRIPCI" in str(celda.value).upper()
                        and "ACTIVIDAD" in str(celda.value).upper()
                    ):
                        fila_encabezado = celda.row
                        break
                if fila_encabezado:
                    break

            if fila_encabezado is None:
                raise ValueError(
                    "No se encontró la fila de encabezados en la plantilla."
                )

            col_map = {}
            for celda in ws[fila_encabezado]:
                if celda.value:
                    key = str(celda.value).strip().upper()
                    col_map[key] = celda.column

            fila_inicio = fila_encabezado + 1

            for r in range(fila_inicio, ws.max_row + 1):
                for c in range(1, ws.max_column + 1):
                    ws.cell(row=r, column=c).value = None

            for i, act in enumerate(actividades):
                fila = fila_inicio + i
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)

                for nombre_col_normalizado, valor in datos.items():
                    col_idx = col_map.get(nombre_col_normalizado)
                    if col_idx:
                        celda = ws.cell(row=fila, column=col_idx, value=valor)
                        if "AUTONOM" in nombre_col_normalizado:
                            celda.number_format = "0%"
                        if nombre_col_normalizado in (
                            "NO.", "FRECUENCIA", "VOL.(SEGÚN FRECUENCIA)",
                            "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                            "AUTONOMÍA DE LA TAREA (0-100%)",
                            "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                        ):
                            celda.alignment = Alignment(
                                horizontal="center", vertical="center"
                            )
                        else:
                            celda.alignment = Alignment(
                                vertical="center", wrap_text=True
                            )

        else:
            ENCABEZADOS = [
                "NO.",
                "DESCRIPCIÓN DE LA ACTIVIDAD",
                "UNIDAD DE MEDIDA",
                "FRECUENCIA",
                "VOL.(SEGÚN FRECUENCIA)",
                "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                "PROCESO DEL ÁREA",
                "AUTONOMÍA DE LA TAREA (0-100%)",
                "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                "VICEPRESIDENCIA",
                "OBSERVACIONES",
            ]
            ANCHOS = [7, 55, 22, 18, 22, 22, 22, 18, 18, 20, 25]

            borde = Border(
                top    = Side(style="thin"),
                bottom = Side(style="thin"),
                left   = Side(style="thin"),
                right  = Side(style="thin"),
            )

            wb = Workbook()
            ws = wb.active
            ws.title = "FORMATO"

            ws.append(ENCABEZADOS)
            for col_idx, _ in enumerate(ENCABEZADOS, start=1):
                c = ws.cell(row=1, column=col_idx)
                c.font      = Font(bold=True, size=14)
                c.alignment = Alignment(
                    horizontal="center", vertical="center", wrap_text=True
                )
                c.border    = borde
            ws.row_dimensions[1].height = 59

            for col_idx, ancho in enumerate(ANCHOS, start=1):
                ws.column_dimensions[get_column_letter(col_idx)].width = ancho

            for i, act in enumerate(actividades):
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)
                fila_vals = [datos.get(enc.upper(), "") for enc in ENCABEZADOS]
                ws.append(fila_vals)
                fila_actual = ws.max_row
                for col_idx in range(1, len(ENCABEZADOS) + 1):
                    c = ws.cell(row=fila_actual, column=col_idx)
                    c.border    = borde
                    c.alignment = Alignment(vertical="center", wrap_text=True)
                    if col_idx == 8:
                        c.number_format = "0%"

        buffer = BytesIO()
        wb.save(buffer)
        buffer.seek(0)
        excel_bytes = buffer.getvalue()

        from datetime import datetime as _dt
        nombre_colaborador = re.sub(
            r"[^a-zA-Z0-9]", "_",
            contexto.get("nombre_completo", "levantamiento") if contexto else "levantamiento"
        ).strip("_") or "levantamiento"
        ts = _dt.now().strftime("%Y%m%d_%H%M%S")
        nombre_archivo = f"levantamiento_{nombre_colaborador}_{ts}.xlsx"

        with open(nombre_archivo, "wb") as f:
            f.write(excel_bytes)

        try:
            from google.colab import files as _colab_files
            _colab_files.download(nombre_archivo)
            print(f"\n📥 Descarga iniciada en tu navegador: {nombre_archivo}")
        except ModuleNotFoundError:
            try:
                from IPython.display import FileLink, display as _display
                _display(FileLink(
                    nombre_archivo,
                    result_html_prefix="📥 Haz clic para descargar tu Excel: "
                ))
            except Exception:
                print(f"\n📥 Excel guardado en: {os.path.abspath(nombre_archivo)}")
        except Exception:
            print(f"\n📥 Excel guardado en: {os.path.abspath(nombre_archivo)}")

        return excel_bytes, nombre_archivo

    # -----------------------------------------------------------------------
    # I. CALCULAR RESUMEN ANALÍTICO
    # -----------------------------------------------------------------------
    def calcular_y_guardar_resumen_bq(
        self,
        actividades,
        metricas,
        contexto,
    ):
        from datetime import datetime, timezone

        JORNADA_LABORAL_DIARIA = 8.5

        total_acts = len(actividades)

        conteo_phva: dict = {}
        for act in actividades:
            p = act.get("phva", "Sin clasificar")
            conteo_phva[p] = conteo_phva.get(p, 0) + 1

        phva_dominante = max(conteo_phva, key=conteo_phva.get) if conteo_phva else ""
        por_phva = round(conteo_phva.get(phva_dominante, 0) / total_acts * 100, 2) \
            if total_acts > 0 else 0.0

        conteo_frec: dict = {}
        for act in actividades:
            f = act.get("frecuencia", "")
            conteo_frec[f] = conteo_frec.get(f, 0) + 1

        def _pct_frec(frec_nombre):
            return round(conteo_frec.get(frec_nombre, 0) / total_acts * 100, 2) \
                if total_acts > 0 else 0.0

        pct_diario     = _pct_frec("Diario")
        pct_semanal    = _pct_frec("Semanal")
        pct_quincenal  = _pct_frec("Quincenal")
        pct_mensual    = _pct_frec("Mensual")
        pct_trimestral = _pct_frec("Trimestral")
        por_pareto     = round(
            pct_diario + pct_semanal + pct_quincenal + pct_mensual + pct_trimestral, 2
        )

        frec_critica = max(conteo_frec, key=conteo_frec.get) if conteo_frec else ""
        cant_frec_critica = conteo_frec.get(frec_critica, 0)
        por_frecuencia = round(cant_frec_critica / total_acts * 100, 2) \
            if total_acts > 0 else 0.0

        # Unidades de medida únicas (orden de primera aparición), máx. 3
        unidades_vistas = []
        for act in actividades:
            u = act.get("unidad_medida", "").strip()
            if u and u not in unidades_vistas:
                unidades_vistas.append(u)
        actividad_1 = unidades_vistas[0] if len(unidades_vistas) > 0 else ""
        actividad_2 = unidades_vistas[1] if len(unidades_vistas) > 1 else ""
        actividad_3 = unidades_vistas[2] if len(unidades_vistas) > 2 else ""

        carga_total = metricas.get("total_carga_w_sin_tm", 0.0)
        por_actividad = round(carga_total * 100, 2)

        if carga_total <= 0.70:
            estado = "Subutilizado"
        elif carga_total <= 1.10:
            estado = "Normal"
        else:
            estado = "Sobrecargado"

        recomendacion_dotacion = metricas.get("numero_personas_requeridas", 1.0)

        print("\n🤖 Generando observaciones y análisis del levantamiento...")
        prompt_obs = f"""
Eres un analista senior de Levantamiento de Cargas Laborales.

Con base en el siguiente resumen de métricas de un colaborador, redacta
un párrafo breve de OBSERVACIONES (máximo 3 oraciones) y una
RECOMENDACIÓN DE DOTACIÓN justificada (1 oración).

Contexto:
- Cargo              : {contexto.get("cargo", "")}
- Gerencia           : {contexto.get("gerencia", "")}
- Vicepresidencia    : {contexto.get("vicepresidencia", "")}
- Total actividades  : {total_acts}
- Carga total (CW)   : {round(carga_total, 4)}
- Estado             : {estado}
- PHVA dominante     : {phva_dominante} ({por_phva}% de actividades)
- Frecuencia crítica : {frec_critica} ({por_frecuencia}% de actividades)
- % Pareto (D+S+Q+M+T): {por_pareto}%
- Dotación calculada : {round(recomendacion_dotacion, 2)} personas

Actividades registradas:
{json.dumps(
    [{"nombre": a.get("nombre"), "frecuencia": a.get("frecuencia"),
      "phva": a.get("phva"), "autonomia": a.get("autonomia")}
     for a in actividades],
    indent=2, ensure_ascii=False
)}

Responde EXCLUSIVAMENTE con un JSON válido:
{{
  "observaciones": "Párrafo de máximo 3 oraciones.",
  "recomendacion_dotacion_texto": "Una oración justificando la dotación."
}}
"""
        observaciones_texto = ""
        rec_dotacion_texto  = ""
        try:
            reply = llm.invoke(prompt_obs)
            resultado_llm = self._parse_json_seguro(reply.content)
            observaciones_texto = resultado_llm.get("observaciones", "")
            rec_dotacion_texto  = resultado_llm.get("recomendacion_dotacion_texto", "")
        except Exception as e:
            observaciones_texto = f"No se pudieron generar observaciones automáticas: {e}"
            rec_dotacion_texto  = ""

        fecha_carga = datetime.now(timezone.utc).strftime("%Y-%m-%d")

        fila_bq = {
            "gerencia":               contexto.get("gerencia", ""),
            "cargo":                  contexto.get("cargo", ""),
            "carga_total":            round(float(carga_total), 4),
            "actividades_phva":       phva_dominante,
            "por_phva":               por_phva,
            "diario":                 pct_diario,
            "semanal":                pct_semanal,
            "quincenal":              pct_quincenal,
            "mensual":                pct_mensual,
            "trimestral":             pct_trimestral,
            "por_pareto":             por_pareto,
            "frecuencia_critica":     frec_critica,
            "cantidad_actividades":   cant_frec_critica,
            "total_actividades":      total_acts,
            "por_frecuencia":         por_frecuencia,
            "actividad_1":            actividad_1,
            "actividad_2":            actividad_2,
            "actividad_3":            actividad_3,
            "estado":                 estado,
            "por_actividad":          por_actividad,
            "observaciones":          observaciones_texto,
            "recomendacion_dotacion": round(float(recomendacion_dotacion), 2),
            "fecha_carga":            fecha_carga,
        }

        # El resumen analítico es para uso interno del analista.
        # No se imprime en pantalla; el analista lo consulta a través
        # del dict retornado (resumen_bq) o via BigQuery.
        return fila_bq

    # -----------------------------------------------------------------------
    def guardar_en_bigquery(
        self,
        actividades,
        contexto,
        project_id,
        dataset_id,
        table_id,
        credentials_path=None
    ):
        from google.cloud import bigquery
        from google.oauth2 import service_account
        from datetime import datetime, timezone

        if credentials_path:
            creds  = service_account.Credentials.from_service_account_file(
                credentials_path,
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            client = bigquery.Client(project=project_id, credentials=creds)
        else:
            client = bigquery.Client(project=project_id)

        tabla_ref = f"{project_id}.{dataset_id}.{table_id}"
        timestamp = datetime.now(timezone.utc).isoformat()

        filas = []
        for act in actividades:
            filas.append({
                "timestamp_carga":       timestamp,
                "cargo":                 contexto.get("cargo", ""),
                "vicepresidencia":       contexto.get("vicepresidencia", ""),
                "gerencia":              contexto.get("gerencia", ""),
                "descripcion_actividad": act.get("nombre", ""),
                "frecuencia":            act.get("frecuencia", ""),
                "volumen":               act.get("volumen"),
                "duracion_min":          act.get("duracion_min"),
                "autonomia_pct":         act.get("autonomia"),
                "proceso_area":          act.get("proceso_area", ""),
                "unidad_medida":         act.get("unidad_medida", ""),
                "phva":                  act.get("phva", ""),
                "observaciones":         act.get("observaciones", ""),
            })

        errores = client.insert_rows_json(tabla_ref, filas)

        if not errores:
            print(f"\n✅ {len(filas)} fila(s) insertada(s) en {tabla_ref}")
        else:
            print(f"\n⚠️ Errores al insertar en BigQuery:")
            for err in errores:
                print(f"   {err}")

    # -----------------------------------------------------------------------
    # F. PIPELINE COMPLETO PARA ACTIVIDADES CONVERSACIONALES
    # -----------------------------------------------------------------------
    def procesar_actividades_conversacional(
        self,
        ruta_excel_planta="Corte_Planta_30Ene2026_Organigrama.xlsx",
        ruta_plantilla_excel=None,
    ):
        # PASO 0 — Selección de modo
        modo = self._seleccionar_modo_inicio()

        # PASO 1 — Capturar contexto del colaborador (búsqueda en Excel)
        contexto = self.capturar_contexto_conversacional(
            ruta_excel_planta=ruta_excel_planta
        )

        # PASO 2 (opcional) — Cargar transcripción de Teams
        actividades_vtt_completadas = []
        if modo == "con_transcripcion":
            actividades_vtt_completadas = self._cargar_vtt_conversacional(
                contexto             = contexto,
                procesos_registrados = [],
            )
            if actividades_vtt_completadas:
                print(
                    f"\nℹ️  {len(actividades_vtt_completadas)} actividad(es) de la "
                    "transcripción se unirán a las que registres manualmente."
                )

        # PASO 3 — Captura conversacional
        procesos_de_vtt = [
            a.get("proceso_area", "")
            for a in actividades_vtt_completadas
            if a.get("proceso_area")
        ]
        actividades_base = self._capturar_actividades_conversacional_con_procesos(
            contexto           = contexto,
            procesos_iniciales = procesos_de_vtt,
        )

        # PASO 4 — Consolidar
        todas_las_actividades_base = actividades_vtt_completadas + actividades_base

        if not todas_las_actividades_base:
            print("\n⚠️ No se registraron actividades.")
            return []

        # PASO 5 — Enriquecimiento IA
        print("\n🤖 Enriqueciendo actividades con IA...")
        actividades_ia = self.enriquecer_actividades(
            todas_las_actividades_base,
            contexto["cargo"],
            contexto["vicepresidencia"],
        )

        # PASO 6 — Fusión
        actividades_finales = self.fusionar_inputs_usuario_nodiarias(
            todas_las_actividades_base, actividades_ia
        )

        # PASO 7 — Resumen global y confirmación final (sin editor)
        print("\n📋 Generando resumen para tu confirmación final...")
        resumen = self.resumen_para_confirmacion(actividades_finales, contexto)
        self.confirmar_informacion(resumen)

        # PASO 8 — Generar Excel descargable
        self.generar_excel_actividades(
            actividades    = actividades_finales,
            contexto       = contexto,
            ruta_plantilla = ruta_plantilla_excel,
        )

        # PASO 9 — Calcular métricas finales y resumen analítico
        metricas_finales = self.calcular_metricas(actividades_finales)
        resumen_bq = self.calcular_y_guardar_resumen_bq(
            actividades = actividades_finales,
            metricas    = metricas_finales,
            contexto    = contexto,
        )

        return {
            "actividades": actividades_finales,
            "contexto":    contexto,
            "resumen_bq":  resumen_bq,
        }


# ---------------------------------------------------------------------------
# Función auxiliar (module-level)
# ---------------------------------------------------------------------------
def _actividad_a_fila(act, numero=None, vicepresidencia=""):
    autonomia_raw = act.get("autonomia")
    autonomia_decimal = (autonomia_raw / 100) if autonomia_raw is not None else ""

    return {
        "NO.":                                   numero if numero is not None else "",
        "DESCRIPCIÓN DE LA ACTIVIDAD":           act.get("nombre", ""),
        "UNIDAD DE MEDIDA":                      act.get("unidad_medida", ""),
        "FRECUENCIA":                            act.get("frecuencia", ""),
        "VOL.(SEGÚN FRECUENCIA)":                act.get("volumen", ""),
        "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS": act.get("duracion_min", ""),
        "PROCESO DEL ÁREA":                      act.get("proceso_area", ""),
        "AUTONOMÍA DE LA TAREA (0-100%)":        autonomia_decimal,
        "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)":   act.get("phva", ""),
        "VICEPRESIDENCIA":                       vicepresidencia,
        "OBSERVACIONES":                         act.get("observaciones", ""),
    }

#### Parte del Usuario

In [10]:
procesador = ProcesadorTranscripcionTeams()

In [11]:
resultado = procesador.procesar_actividades_conversacional()

# Extraer correctamente
actividades = resultado["actividades"]
contexto    = resultado["contexto"]
resumen_bq  = resultado["resumen_bq"]

# Ahora sí funciona:
metricas = procesador.calcular_metricas(actividades)


════════════════════════════════════════════════════════════
  🚀  BIENVENIDO — LEVANTAMIENTO DE CARGAS LABORALES
════════════════════════════════════════════════════════════

¿Cómo deseas iniciar el proceso?

  1) Solo registrar mis actividades manualmente
  2) Registrar actividades y además cargar la transcripción
     de una reunión de Microsoft Teams (archivo .vtt)



Selecciona una opción (1 / 2):  1



✅ Modo seleccionado: Registro manual de actividades.


════════════════════════════════════════════════════════════
  👤  IDENTIFICACIÓN DEL COLABORADOR
════════════════════════════════════════════════════════════

  ✅ Planta cargada: 10797 colaborador(es) registrado(s).



  Ingresa tu número de documento (sin puntos ni espacios):  39779089



────────────────────────────────────────────────────────────
  ✅  Colaborador encontrado:
────────────────────────────────────────────────────────────
  Nombre          : GLADYS MARIA
  Cargo           : 10010-Analista I
  Vicepresidencia : 9929 VP DE TALENTO Y ADMIN
  Gerencia        : 9263 GCIA EXPERIENCIA Y COMUNICACIONES
────────────────────────────────────────────────────────────


...

¿Tienes otra actividad para registrar? (si / no):  no



════════════════════════════════════════════════════════════
  ✅  Se registraron 4 actividad(es) en total.
════════════════════════════════════════════════════════════

🤖 Enriqueciendo actividades con IA...

📋 Generando resumen para tu confirmación final...

════════════════════════════════════════════════════════════
  📋  RESUMEN FINAL DEL LEVANTAMIENTO
════════════════════════════════════════════════════════════
Resumen de actividades del colaborador 10010-Analista I en la VP de Talento y Admin:

1. Publicación tasas internacional  
- Descripción: Actualizar diariamente en la intranet las tasas que se reciben desde el área internacional y dejarlas visibles para todo el banco.  
- Frecuencia: Diario  
- Duración: 15 minutos  
- Volumen: 1  
- Unidad de medida: publicaciones  
- Tipo de actividad: Hacer  
- Autonomía: 100%

2. Publicación actualizaciones intranet  
- Descripción: Revisar semanalmente que los enlaces funcionen, corregir ortografía y publicar las actualizaciones hechas 


¿Confirmas que la información es correcta? (si / no):  si


/home/jupyter/bbog-geo-cargas-laborales-mll/app/notebooks/levantamiento_GLADYS_MARIA_20260223_211522.xlsx


🤖 Generando observaciones y análisis del levantamiento...


In [12]:
from load_BigQuery import insertar_resumen

insertar_resumen(resumen_bq)

✅ Resumen insertado en bdb-gcp-sbx-ia.wa_test.TBL_GEO_IA_CARGAS_LABORALES
   Gerencia    : 9263 GCIA EXPERIENCIA Y COMUNICACIONES
   Cargo : 10010-Analista I
   Carga Total : 0.0605
   Actividades PHVA : Hacer
   Porcentaje PHVA : 50.0
   Diario : 25.0
   Semanal : —
   Quincenal : 0.0
   Mensual : 50.0
   Trimestral : 0.0
   Porcentaje Pareto : 100.0
   Frecuencia Critica : Mensual
   Cantidad Actividades : 2
   Total Actividades : 4
   Porcentaje Frecuencia : 50.0
   Actividad_1 : publicaciones
   Actividad_2 : revisiones
   Actividad_3 : actualizaciones
   Estado      : Subutilizado
   Porcentaje Actividad : 6.05
   Observaciones : El colaborador presenta una carga laboral muy baja, con una dotación calculada de solo 0.1 personas, lo que indica una subutilización significativa. Las actividades realizadas son principalmente de tipo 'Hacer' y con frecuencia mensual o menor, lo que contribuye a la baja carga. Además, todas las tareas cuentan con autonomía completa, lo que sugiere que e

#### Parte del Analista

In [13]:
"""
WorkloadAnalysisAgent - Agente de IA para Levantamiento de Cargas
=================================================================
Lee datos pre-calculados desde BigQuery, permite al analista elegir
la Gerencia interactivamente y genera el análisis completo en un
archivo Word (.docx) descargable desde Jupyter.

Dependencias:
    pip install google-cloud-bigquery pandas numpy python-docx

Uso:
    python workload_analysis_agent.py
    # o en Jupyter:
    # %run workload_analysis_agent.py
"""

from __future__ import annotations
from dataclasses import dataclass, field
import base64
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

# Word document generation
from docx import Document as DocxDocument
from docx.shared import Pt, RGBColor, Inches, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# Jupyter display (solo se usa si estamos en Jupyter)
try:
    from IPython.display import display, HTML, FileLink
    IN_JUPYTER = True
except ImportError:
    IN_JUPYTER = False


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN  ← Ajusta estos valores antes de correr el script
# ═══════════════════════════════════════════════════════════════════════════════

BQ_PROJECT = "bdb-gcp-sbx-ia"
BQ_DATASET = "wa_test"
BQ_TABLE   = "TBL_GEO_IA_CARGAS_LABORALES"

COL = {
    "gerencia":               "gerencia",
    "cargo":                  "cargo",
    "carga_total":            "carga_total",
    "actividades_phva":       "actividades_phva",
    "por_phva":               "por_phva",
    "diario":                 "diario",
    "semanal":                "semanaL",          # ojo mayúscula en tu tabla
    "quincenal":              "quincenal",
    "mensual":                "mensual",
    "trimestral":             "trimestral",
    "por_pareto":             "por_pareto",
    "frecuencia_critica":     "frecuencia_critica",
    "cantidad_actividades":   "cantidad_actividades",
    "total_actividades":      "total_actividades",
    "por_frecuencia":         "por_frecuencia",
    "actividad_1":            "actividad_1",
    "actividad_2":            "actividad_2",
    "actividad_3":            "actividad_3",
    "estado":                 "estado",
    "por_actividad":          "por_actividad",
    "observaciones":          "observaciones",
    "fecha_carga":            "fecha_carga",
    "recomendacion_dotacion": "recomendacion_dotacion",
}

UMBRALES = {
    "sostenible":          85,
    "ajustada":            95,
    "sobrecarga_moderada": 105,
    "sobrecarga":          120,
}

PHVA_ORDER = ["PLANEAR", "HACER", "VERIFICAR", "ACTUAR"]

# Colores del documento Word
COLOR_HEADER_BG  = "2E4057"   # Azul oscuro - encabezados principales
COLOR_SUBHDR_BG  = "048A81"   # Verde azulado - subencabezados
COLOR_ROW_ALT    = "EEF2F7"   # Gris muy claro - filas alternas
COLOR_WHITE      = "FFFFFF"
COLOR_STATUS = {
    "sostenible":          "27AE60",   # verde
    "ajustada":            "F39C12",   # amarillo
    "sobrecarga_moderada": "E67E22",   # naranja
    "sobrecarga":          "E74C3C",   # rojo
    "critica":             "8E1A1A",   # rojo oscuro
}


# ═══════════════════════════════════════════════════════════════════════════════
# Dataclass de cargo
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class CargoData:
    gerencia:               str
    cargo:                  str
    carga_total:            float
    actividades_phva:       str
    por_phva:               float
    diario:                 float
    semanal:                float
    quincenal:              float = 0.0
    mensual:                float = 0.0
    trimestral:             float = 0.0
    por_pareto:             float = 0.0
    frecuencia_critica:     str   = ""
    cantidad_actividades:   int   = 0
    total_actividades:      int   = 0
    por_frecuencia:         float = 0.0
    actividad_1:            str   = ""
    actividad_2:            str   = ""
    actividad_3:            str   = ""
    estado:                 str   = ""
    por_actividad:          float = 0.0
    observaciones:          str   = ""
    fecha_carga:            str   = ""
    recomendacion_dotacion: str   = ""


# ═══════════════════════════════════════════════════════════════════════════════
# Capa BigQuery
# ═══════════════════════════════════════════════════════════════════════════════

class BigQueryLoader:
    def __init__(self, project: str, dataset: str, table: str):
        self.project    = project
        self.dataset    = dataset
        self.table      = table
        self.full_table = f"`{project}.{dataset}.{table}`"
        self.client     = bigquery.Client(project=project)

    def list_gerencias(self) -> list[str]:
        col = COL["gerencia"]
        query = f"""
            SELECT DISTINCT `{col}`
            FROM {self.full_table}
            WHERE `{col}` IS NOT NULL
            ORDER BY `{col}`
        """
        return [row[0] for row in self.client.query(query).result()]

    def load_gerencia(self, gerencia: str) -> pd.DataFrame:
        col_g = COL["gerencia"]
        cols  = ", ".join(f"`{v}` AS `{k}`" for k, v in COL.items())
        query = f"""
            SELECT {cols}
            FROM {self.full_table}
            WHERE `{col_g}` = @gerencia
            ORDER BY `{COL['cargo']}`
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("gerencia", "STRING", gerencia)]
        )
        df = self.client.query(query, job_config=job_config).to_dataframe()
        if df.empty:
            raise ValueError(f"No se encontraron datos para la gerencia '{gerencia}'.")
        return df

    def df_to_cargos(self, df: pd.DataFrame) -> list[CargoData]:
        cargos = []
        for _, row in df.iterrows():
            def fval(col, default=0.0):
                v = row.get(col, default)
                try:   return float(v) if pd.notna(v) else default
                except: return default

            def sval(col, default=""):
                v = row.get(col, default)
                return str(v).strip() if pd.notna(v) else default

            def ival(col, default=0):
                v = row.get(col, default)
                try:   return int(v) if pd.notna(v) else default
                except: return default

            cargos.append(CargoData(
                gerencia=               sval("gerencia"),
                cargo=                  sval("cargo"),
                carga_total=            fval("carga_total"),
                actividades_phva=       sval("actividades_phva"),
                por_phva=               fval("por_phva"),
                diario=                 fval("diario"),
                semanal=                fval("semanal"),
                quincenal=              fval("quincenal"),
                mensual=                fval("mensual"),
                trimestral=             fval("trimestral"),
                por_pareto=             fval("por_pareto"),
                frecuencia_critica=     sval("frecuencia_critica"),
                cantidad_actividades=   ival("cantidad_actividades"),
                total_actividades=      ival("total_actividades"),
                por_frecuencia=         fval("por_frecuencia"),
                actividad_1=            sval("actividad_1"),
                actividad_2=            sval("actividad_2"),
                actividad_3=            sval("actividad_3"),
                estado=                 sval("estado"),
                por_actividad=          fval("por_actividad"),
                observaciones=          sval("observaciones"),
                fecha_carga=            sval("fecha_carga"),
                recomendacion_dotacion= sval("recomendacion_dotacion"),
            ))
        return cargos


# ═══════════════════════════════════════════════════════════════════════════════
# Helpers para python-docx
# ═══════════════════════════════════════════════════════════════════════════════

def _set_cell_bg(cell, hex_color: str):
    """Aplica color de fondo a una celda de tabla."""
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement("w:shd")
    shd.set(qn("w:val"),   "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"),  hex_color)
    tcPr.append(shd)


def _set_cell_borders(cell, color: str = "CCCCCC"):
    """Aplica bordes finos a una celda."""
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = OxmlElement("w:tcBorders")
    for side in ("top", "left", "bottom", "right"):
        border = OxmlElement(f"w:{side}")
        border.set(qn("w:val"),   "single")
        border.set(qn("w:sz"),    "4")
        border.set(qn("w:space"), "0")
        border.set(qn("w:color"), color)
        tcBorders.append(border)
    tcPr.append(tcBorders)


def _bold_run(para, text: str, font_size: int = 11, color_hex: str = None):
    run = para.add_run(text)
    run.bold = True
    run.font.size = Pt(font_size)
    if color_hex:
        run.font.color.rgb = RGBColor.from_string(color_hex)
    return run


def _add_section_heading(doc: DocxDocument, text: str, level: int = 1):
    """Agrega un encabezado con estilo corporativo."""
    para = doc.add_paragraph()
    para.paragraph_format.space_before = Pt(14 if level == 1 else 8)
    para.paragraph_format.space_after  = Pt(4)
    run = para.add_run(text)
    run.bold = True
    run.font.size = Pt(14 if level == 1 else 12)
    run.font.color.rgb = RGBColor.from_string(COLOR_HEADER_BG if level == 1 else COLOR_SUBHDR_BG)
    # Línea inferior decorativa
    pPr  = para._p.get_or_add_pPr()
    pBdr = OxmlElement("w:pBdr")
    bdr  = OxmlElement("w:bottom")
    bdr.set(qn("w:val"),   "single")
    bdr.set(qn("w:sz"),    "6")
    bdr.set(qn("w:space"), "1")
    bdr.set(qn("w:color"), COLOR_HEADER_BG if level == 1 else COLOR_SUBHDR_BG)
    pBdr.append(bdr)
    pPr.append(pBdr)
    return para


def _status_color(pct: float) -> str:
    if pct <= UMBRALES["sostenible"]:          return COLOR_STATUS["sostenible"]
    if pct <= UMBRALES["ajustada"]:            return COLOR_STATUS["ajustada"]
    if pct <= UMBRALES["sobrecarga_moderada"]: return COLOR_STATUS["sobrecarga_moderada"]
    if pct <= UMBRALES["sobrecarga"]:          return COLOR_STATUS["sobrecarga"]
    return COLOR_STATUS["critica"]


def _add_kv_table(doc: DocxDocument, rows: list[tuple[str, str]]):
    """Tabla de dos columnas Indicador | Valor."""
    tbl = doc.add_table(rows=0, cols=2)
    tbl.style = "Table Grid"
    col_widths = [Inches(2.8), Inches(4.2)]

    for i, (k, v) in enumerate(rows):
        tr = tbl.add_row()
        bg = COLOR_ROW_ALT if i % 2 == 0 else COLOR_WHITE

        # Indicador
        c0 = tr.cells[0]
        c0.width = col_widths[0]
        _set_cell_bg(c0, bg)
        _set_cell_borders(c0)
        p0 = c0.paragraphs[0]
        r0 = p0.add_run(k)
        r0.bold = True
        r0.font.size = Pt(9.5)
        r0.font.color.rgb = RGBColor.from_string("2E4057")
        c0.vertical_alignment = WD_ALIGN_VERTICAL.CENTER

        # Valor
        c1 = tr.cells[1]
        c1.width = col_widths[1]
        _set_cell_bg(c1, bg)
        _set_cell_borders(c1)
        p1 = c1.paragraphs[0]
        r1 = p1.add_run(str(v))
        r1.font.size = Pt(9.5)
        c1.vertical_alignment = WD_ALIGN_VERTICAL.CENTER

    doc.add_paragraph()  # espacio posterior


def _add_data_table(doc: DocxDocument, headers: list[str], data_rows: list[list]):
    """Tabla genérica con encabezado coloreado y filas alternas."""
    if not data_rows:
        doc.add_paragraph("(Sin datos)").italic = True
        return

    tbl = doc.add_table(rows=0, cols=len(headers))
    tbl.style = "Table Grid"

    # Encabezado
    hdr_row = tbl.add_row()
    for i, h in enumerate(headers):
        cell = hdr_row.cells[i]
        _set_cell_bg(cell, COLOR_SUBHDR_BG)
        _set_cell_borders(cell, COLOR_SUBHDR_BG)
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = p.add_run(h)
        run.bold = True
        run.font.size = Pt(9.5)
        run.font.color.rgb = RGBColor.from_string(COLOR_WHITE)

    # Filas de datos
    for i, row in enumerate(data_rows):
        bg  = COLOR_ROW_ALT if i % 2 == 0 else COLOR_WHITE
        tr  = tbl.add_row()
        for j, val in enumerate(row):
            cell = tr.cells[j]
            _set_cell_bg(cell, bg)
            _set_cell_borders(cell)
            p = cell.paragraphs[0]
            p.add_run(str(val)).font.size = Pt(9.5)

    doc.add_paragraph()


def _add_bullet_list(doc: DocxDocument, items: list[str], color_hex: str = "2E4057"):
    for item in items:
        para = doc.add_paragraph(style="List Bullet")
        para.paragraph_format.space_after = Pt(2)
        run = para.add_run(item)
        run.font.size = Pt(10)
        run.font.color.rgb = RGBColor.from_string(color_hex)


def _add_recommendation_box(doc: DocxDocument, text: str):
    """Cuadro destacado para la recomendación."""
    tbl = doc.add_table(rows=1, cols=1)
    tbl.style = "Table Grid"
    cell = tbl.cell(0, 0)
    _set_cell_bg(cell, "EAF4FB")
    _set_cell_borders(cell, "048A81")
    p = cell.paragraphs[0]
    run = p.add_run("💡 " + text)
    run.font.size = Pt(10)
    run.font.color.rgb = RGBColor.from_string("1A5276")
    doc.add_paragraph()


# ═══════════════════════════════════════════════════════════════════════════════
# Motor de análisis
# ═══════════════════════════════════════════════════════════════════════════════

class WorkloadAnalysisAgent:
    """
    Agente de análisis de carga laboral.
    Genera un archivo Word (.docx) con el informe completo.
    """

    def __init__(self, cargos: list[CargoData], gerencia_name: str = ""):
        self.cargos        = cargos
        self.gerencia_name = gerencia_name

    # ── Clasificación ─────────────────────────────────────────────────────────

    @staticmethod
    def _classify(pct: float) -> str:
        if pct <= UMBRALES["sostenible"]:          return "Carga sostenible"
        if pct <= UMBRALES["ajustada"]:            return "Carga ajustada (riesgo leve)"
        if pct <= UMBRALES["sobrecarga_moderada"]: return "Sobrecarga moderada"
        if pct <= UMBRALES["sobrecarga"]:          return "Sobrecarga"
        return                                            "Sobrecarga crítica"

    # ── Frecuencias ───────────────────────────────────────────────────────────

    def _frequency_rows(self, c: CargoData) -> list[list]:
        freqs = [
            ("Diario",     c.diario),
            ("Semanal",    c.semanal),
            ("Quincenal",  c.quincenal),
            ("Mensual",    c.mensual),
            ("Trimestral", c.trimestral),
        ]
        rows  = [[f, f"{round(v,1)}%"] for f, v in freqs if v > 0]
        total = sum(v for _, v in freqs if v > 0)
        rows.append(["TOTAL", f"{round(total,1)}%"])
        return rows

    # ── Actividades ───────────────────────────────────────────────────────────

    def _activity_rows(self, c: CargoData) -> list[list]:
        acts = [a for a in [c.actividad_1, c.actividad_2, c.actividad_3] if a]
        return [[i + 1, a] for i, a in enumerate(acts)]

    # ── Puntos de atención ────────────────────────────────────────────────────

    def _attention_points(self, c: CargoData) -> list[str]:
        pts = []
        if c.por_pareto > 70:
            pts.append(
                f"Alta presión en frecuencias cortas: el {round(c.por_pareto,1)}% de la carga "
                f"proviene de actividades Diarias y Semanales."
            )
        if c.frecuencia_critica:
            pts.append(
                f"Frecuencia crítica identificada: '{c.frecuencia_critica}' concentra "
                f"la mayor parte de la carga operativa."
            )
        if c.carga_total > UMBRALES["sobrecarga_moderada"]:
            pts.append(
                f"Carga total de {round(c.carga_total,1)}% supera el umbral de sobrecarga moderada "
                f"({UMBRALES['sobrecarga_moderada']}%): riesgo de saturación."
            )
        elif c.carga_total > UMBRALES["ajustada"]:
            pts.append(
                f"Carga total de {round(c.carga_total,1)}% se encuentra en zona de ajuste. "
                "Monitorear para evitar que escale a sobrecarga."
            )
        if c.por_phva > 60:
            pts.append(
                f"El {round(c.por_phva,1)}% de la carga está concentrada en una sola fase PHVA. "
                "Revisar balance entre planeación, ejecución y verificación."
            )
        if c.observaciones:
            pts.append(f"Observación registrada: {c.observaciones}")
        return pts or ["Sin puntos de atención críticos identificados."]

    # ── Impacto en el equipo ──────────────────────────────────────────────────

    def _team_impact(self, c: CargoData) -> list[str]:
        impacts = []
        pct = c.carga_total
        if pct > UMBRALES["sobrecarga_moderada"]:
            impacts.append(
                f"Posición sobrecargada ({self._classify(pct)}): riesgo de errores por saturación "
                "y falta de espacio para mejora de procesos."
            )
        if c.por_pareto > 70:
            impacts.append(
                "Rol con alta dependencia de rutinas diarias/semanales. "
                "Potencial cuello de botella para el área."
            )
        if pct > UMBRALES["sobrecarga"]:
            impacts.append(
                "Riesgo de rotación o ausentismo por agotamiento. "
                "Considerar redistribución de tareas o incorporación de un apoyo."
            )
        if c.total_actividades > 0:
            pct_team = round(c.cantidad_actividades / c.total_actividades * 100, 1)
            if pct_team > 40:
                impacts.append(
                    f"Este cargo concentra el {pct_team}% de las actividades totales del equipo. "
                    "Alta dependencia estructural del área en este rol."
                )
        return impacts or ["Carga dentro de rangos aceptables. Sin impacto negativo proyectado."]

    # ── Recomendación ─────────────────────────────────────────────────────────

    def _recommendation(self, c: CargoData) -> str:
        if c.recomendacion_dotacion:
            return c.recomendacion_dotacion
        pct = c.carga_total
        if pct > UMBRALES["sobrecarga_moderada"]:
            if c.por_pareto > 65:
                return (
                    f"Delegar o automatizar el 30% de las actividades '{c.frecuencia_critica or 'diarias'}' "
                    "de mayor volumen para liberar capacidad, reenfocando el tiempo hacia "
                    "planificación estratégica (PLANEAR) y ajuste de procesos (ACTUAR)."
                )
            return (
                "Establecer una 'Revisión Semanal de Impacto' (fase VERIFICAR/ACTUAR) "
                "redistribuyendo tiempo de actividades de alta frecuencia."
            )
        if pct > UMBRALES["ajustada"]:
            return (
                "Redistribuir 1 de las actividades principales a otro miembro del equipo "
                "o automatizarla para alcanzar una carga sostenible."
            )
        return (
            "Carga dentro de rangos óptimos. Mantener seguimiento mensual y "
            "explorar oportunidades de mejora continua en fases VERIFICAR y ACTUAR."
        )

    # ── Análisis completo por cargo ───────────────────────────────────────────

    def analyze_cargo(self, c: CargoData) -> dict:
        return {
            "cargo":             c.cargo,
            "carga_total":       c.carga_total,
            "status":            self._classify(c.carga_total),
            "freq_rows":         self._frequency_rows(c),
            "act_rows":          self._activity_rows(c),
            "puntos_atencion":   self._attention_points(c),
            "impacto_equipo":    self._team_impact(c),
            "recomendacion":     self._recommendation(c),
            "raw":               c,
        }

    # ── Resumen del equipo ────────────────────────────────────────────────────

    def build_team_summary_rows(self) -> list[list]:
        rows = []
        for c in self.cargos:
            pct_team = (
                round(c.cantidad_actividades / c.total_actividades * 100, 1)
                if c.total_actividades > 0 else 0
            )
            rows.append([
                c.cargo,
                f"{round(c.carga_total,1)}%",
                f"{round(c.por_pareto,1)}%",
                c.frecuencia_critica,
                f"{round(c.por_phva,1)}%",
                str(c.cantidad_actividades),
                f"{pct_team}%",
                self._classify(c.carga_total),
                c.fecha_carga,
            ])
        return rows

    # ── Portada del documento ─────────────────────────────────────────────────

    def _build_cover(self, doc: DocxDocument):
        doc.add_paragraph()
        doc.add_paragraph()

        title = doc.add_paragraph()
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = title.add_run("ANÁLISIS DE CARGA LABORAL")
        run.bold = True
        run.font.size = Pt(22)
        run.font.color.rgb = RGBColor.from_string(COLOR_HEADER_BG)

        sub = doc.add_paragraph()
        sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run2 = sub.add_run(f"Gerencia: {self.gerencia_name}")
        run2.bold = True
        run2.font.size = Pt(14)
        run2.font.color.rgb = RGBColor.from_string(COLOR_SUBHDR_BG)

        doc.add_paragraph()

        meta = doc.add_paragraph()
        meta.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run3 = meta.add_run(f"Generado: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
        run3.font.size = Pt(10)
        run3.font.color.rgb = RGBColor.from_string("888888")

        doc.add_page_break()

    # ── Sección de un cargo ───────────────────────────────────────────────────

    def _build_cargo_section(self, doc: DocxDocument, a: dict):
        c: CargoData = a["raw"]

        # Título del cargo
        _add_section_heading(doc, f"Cargo: {c.cargo.upper()}", level=1)

        # Estado con color
        status_para = doc.add_paragraph()
        status_para.paragraph_format.space_after = Pt(6)
        label = status_para.add_run("Estado: ")
        label.bold = True
        label.font.size = Pt(11)
        val = status_para.add_run(a["status"] + f"  ({round(c.carga_total,1)}%)")
        val.bold = True
        val.font.size = Pt(11)
        val.font.color.rgb = RGBColor.from_string(_status_color(c.carga_total))

        # KPIs
        _add_section_heading(doc, "Indicadores Clave", level=2)
        kpi_rows = [
            ("Carga total",                   f"{round(c.carga_total,1)}%"),
            ("% Carga por PHVA (concentrado)", f"{round(c.por_phva,1)}%"),
            ("Actividades PHVA",               c.actividades_phva),
            ("Pareto Diario + Semanal",        f"{round(c.por_pareto,1)}%"),
            ("Frecuencia crítica",             c.frecuencia_critica),
            ("Nº de actividades del cargo",    str(c.cantidad_actividades)),
            ("Total actividades del equipo",   str(c.total_actividades)),
            ("% participación por actividad",  f"{round(c.por_actividad,1)}%"),
            ("Estado",                         c.estado),
            ("Fecha de carga",                 c.fecha_carga),
        ]
        _add_kv_table(doc, kpi_rows)

        # Frecuencias
        _add_section_heading(doc, "Distribución por Frecuencia", level=2)
        _add_data_table(doc, ["Frecuencia", "% Carga"], a["freq_rows"])

        # Actividades principales
        if a["act_rows"]:
            _add_section_heading(doc, "Actividades Principales", level=2)
            _add_data_table(doc, ["#", "Actividad Principal"], a["act_rows"])

        # Puntos de atención
        _add_section_heading(doc, "Puntos de Atención", level=2)
        _add_bullet_list(doc, a["puntos_atencion"], color_hex="C0392B")

        # Impacto en el equipo
        _add_section_heading(doc, "Impacto en el Equipo (Gerencia)", level=2)
        _add_bullet_list(doc, a["impacto_equipo"], color_hex="1A5276")

        # Recomendación
        _add_section_heading(doc, "Recomendación", level=2)
        _add_recommendation_box(doc, a["recomendacion"])

        doc.add_page_break()

    # ── Sección resumen del equipo ────────────────────────────────────────────

    def _build_team_section(self, doc: DocxDocument):
        _add_section_heading(doc, "Análisis General del Equipo", level=1)
        headers = [
            "Cargo", "Carga Total", "Pareto D+S", "Frec. Crítica",
            "% PHVA", "# Actividades", "% Equipo", "Estado", "Fecha Carga",
        ]
        _add_data_table(doc, headers, self.build_team_summary_rows())

    # ── Orquestador principal ─────────────────────────────────────────────────

    def run_full_analysis(self, output_path: str = None) -> str:
        """
        Genera el informe Word completo.

        Args:
            output_path: Ruta del archivo de salida. Si es None, se genera
                         automáticamente con timestamp.

        Returns:
            Ruta absoluta del archivo generado.
        """
        if output_path is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            safe_name = self.gerencia_name.replace(" ", "_").replace("/", "-")
            output_path = f"Analisis_Cargas_{safe_name}_{ts}.docx"

        doc = DocxDocument()

        # Márgenes de página (2 cm)
        for section in doc.sections:
            section.top_margin    = Cm(2)
            section.bottom_margin = Cm(2)
            section.left_margin   = Cm(2.5)
            section.right_margin  = Cm(2.5)

        # Fuente base
        doc.styles["Normal"].font.name = "Calibri"
        doc.styles["Normal"].font.size = Pt(10.5)

        # Construir contenido
        self._build_cover(doc)

        cargo_analyses = {c.cargo: self.analyze_cargo(c) for c in self.cargos}
        for a in cargo_analyses.values():
            self._build_cargo_section(doc, a)

        self._build_team_section(doc)

        # Guardar
        doc.save(output_path)
        abs_path = str(Path(output_path).resolve())
        print(f"\n✅  Informe generado: {abs_path}")

        # Enlace de descarga en Jupyter
        if IN_JUPYTER:
            _jupyter_download_link(abs_path, self.gerencia_name)

        return abs_path


# ═══════════════════════════════════════════════════════════════════════════════
# Enlace de descarga para Jupyter
# ═══════════════════════════════════════════════════════════════════════════════

def _jupyter_download_link(filepath: str, gerencia: str):
    """
    Muestra un botón de descarga directamente en la celda de Jupyter.
    Funciona en Jupyter Lab, Jupyter Notebook clásico y VS Code.
    """
    filename = Path(filepath).name

    # Método 1: FileLink (más simple, funciona en Jupyter clásico)
    try:
        link = FileLink(filepath, result_html_prefix="📥 &nbsp;<b>Descargar informe:</b> &nbsp;")
        display(link)
    except Exception:
        pass

    # Método 2: Botón base64 (funciona en entornos donde FileLink no embebe)
    try:
        with open(filepath, "rb") as f:
            data = base64.b64encode(f.read()).decode()

        html = f"""
        <div style="margin: 12px 0; padding: 12px 16px;
                    background: #EAF4FB; border-left: 4px solid #048A81;
                    border-radius: 4px; font-family: Calibri, sans-serif;">
            <span style="font-weight:bold; color:#2E4057;">📄 Informe listo:</span>
            <span style="color:#555;">&nbsp;{filename}</span><br><br>
            <a href="data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{data}"
               download="{filename}"
               style="display:inline-block; padding:8px 18px;
                      background:#2E4057; color:white; font-weight:bold;
                      border-radius:4px; text-decoration:none; font-size:13px;">
               ⬇️ Descargar Word (.docx)
            </a>
            <span style="margin-left:12px; color:#888; font-size:11px;">
                Gerencia: {gerencia}
            </span>
        </div>
        """
        display(HTML(html))
    except Exception as e:
        print(f"(No se pudo generar el enlace HTML de descarga: {e})")


# ═══════════════════════════════════════════════════════════════════════════════
# Punto de entrada interactivo
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("\n" + "=" * 60)
    print("  Agente de Levantamiento de Cargas")
    print("=" * 60)

    # 1. Conectar a BigQuery
    print("\nConectando a BigQuery...")
    try:
        loader    = BigQueryLoader(BQ_PROJECT, BQ_DATASET, BQ_TABLE)
        gerencias = loader.list_gerencias()
    except Exception as e:
        print(f"\n❌  Error al conectar con BigQuery: {e}")
        return

    if not gerencias:
        print("❌  No se encontraron gerencias en la tabla.")
        return

    # 2. Mostrar opciones
    print("\nGerencias disponibles:\n")
    for i, g in enumerate(gerencias, 1):
        print(f"  [{i}] {g}")

    # 3. Input del analista
    print()
    while True:
        entrada = input("Ingresa el nombre o número de la Gerencia a analizar: ").strip()

        if entrada.isdigit():
            idx = int(entrada) - 1
            if 0 <= idx < len(gerencias):
                gerencia_elegida = gerencias[idx]
                break
            print(f"  ⚠  Número fuera de rango. Elige entre 1 y {len(gerencias)}.")
        else:
            matches = [g for g in gerencias if entrada.lower() in g.lower()]
            if len(matches) == 1:
                gerencia_elegida = matches[0]
                break
            elif len(matches) > 1:
                print(f"  ⚠  Varias coincidencias: {matches}. Sé más específico.")
            else:
                print(f"  ⚠  No se encontró '{entrada}'. Intenta de nuevo.")

    # 4. Cargar y analizar
    print(f"\n📥  Cargando datos de: {gerencia_elegida} ...")
    try:
        df     = loader.load_gerencia(gerencia_elegida)
        cargos = loader.df_to_cargos(df)
    except Exception as e:
        print(f"\n❌  Error al cargar datos: {e}")
        return

    print(f"✅  {len(cargos)} cargo(s) encontrados.\n")

    agent = WorkloadAnalysisAgent(cargos=cargos, gerencia_name=gerencia_elegida)
    agent.run_full_analysis()   # genera el .docx y muestra el enlace en Jupyter


if __name__ == "__main__":
    main()


  Agente de Levantamiento de Cargas

Conectando a BigQuery...

Gerencias disponibles:

  [1] 
  [2] 9263 GCIA EXPERIENCIA Y COMUNICACIONES
  [3] 9268 GCIA DE EFECTIVIDAD ORGANIZACIONAL
  [4] Gerencia de Experiencia y Comunicaciones



Ingresa el nombre o número de la Gerencia a analizar:  2



📥  Cargando datos de: 9263 GCIA EXPERIENCIA Y COMUNICACIONES ...
✅  1 cargo(s) encontrados.


✅  Informe generado: /home/jupyter/bbog-geo-cargas-laborales-mll/app/notebooks/Analisis_Cargas_9263_GCIA_EXPERIENCIA_Y_COMUNICACIONES_20260223_212509.docx


/home/jupyter/bbog-geo-cargas-laborales-mll/app/notebooks/Analisis_Cargas_9263_GCIA_EXPERIENCIA_Y_COMUNICACIONES_20260223_212509.docx